# Help Desk — Rapport de suivi

## Analyse de l'activité et des temps de résolution

**Période analysée :** Juin – Septembre 2026

Ce rapport présente une analyse descriptive de l'activité du Help Desk,
des temps de résolution, de leur répartition par client, problématique
et technicien, ainsi que leur distribution temporelle.

> Ce rapport constitue une première étape analytique avant la mise en
> place d'un dashboard interactif Python / Streamlit.

## 1. Configuration et imports

In [285]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import skew

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [286]:
print("Environnement analytique initialisé.")

Environnement analytique initialisé.


## 2. Chargement des données

In [287]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "figures"

DATA_FILE = DATA_DIR / "tickets.csv"

df_raw = pd.read_csv(DATA_FILE)

df_raw.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,SUIVI SLA — HELP DESK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,À compléter quotidiennement par chaque technic...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien,Temps de résolution (min),Client Normalisé,Problématique Normalisée,Date et heure ouverture,Date et Heure résolution,Heure ouverture analytique,Jour ouverture,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Technicien Normalisé
4,23/06/2026,P260604001,Etam,Fermeture provisoire : demande démontage,Faible,23/06/2026 à 9h40,23/06/2026 à 10h13,0h19min,Sofiane AOUES,"19,00",ETAM,Fermeture Provisoire,23/06/2026 9h40,23/06/2026 10h13,9,Dimanche,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sofiane Aoues


## 3. Audit et qualité des données

In [288]:
print(f"Nombre de lignes : {len(df_raw):,}")
print(f"Nombre de colonnes : {len(df_raw.columns)}")

Nombre de lignes : 735
Nombre de colonnes : 26


In [289]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 735 entries, 0 to 734
Data columns (total 26 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   733 non-null    str    
 1   Unnamed: 1   681 non-null    str    
 2   Unnamed: 2   681 non-null    str    
 3   Unnamed: 3   681 non-null    str    
 4   Unnamed: 4   681 non-null    str    
 5   Unnamed: 5   681 non-null    str    
 6   Unnamed: 6   681 non-null    str    
 7   Unnamed: 7   681 non-null    str    
 8   Unnamed: 8   681 non-null    str    
 9   Unnamed: 9   662 non-null    str    
 10  Unnamed: 10  680 non-null    str    
 11  Unnamed: 11  680 non-null    str    
 12  Unnamed: 12  663 non-null    str    
 13  Unnamed: 13  648 non-null    str    
 14  Unnamed: 14  663 non-null    str    
 15  Unnamed: 15  660 non-null    str    
 16  Unnamed: 16  0 non-null      float64
 17  Unnamed: 17  0 non-null      float64
 18  Unnamed: 18  0 non-null      float64
 19  Unnamed: 19  0 non-

In [290]:
df_raw.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,SUIVI SLA — HELP DESK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,À compléter quotidiennement par chaque technic...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien,Temps de résolution (min),Client Normalisé,Problématique Normalisée,Date et heure ouverture,Date et Heure résolution,Heure ouverture analytique,Jour ouverture,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Technicien Normalisé
4,23/06/2026,P260604001,Etam,Fermeture provisoire : demande démontage,Faible,23/06/2026 à 9h40,23/06/2026 à 10h13,0h19min,Sofiane AOUES,"19,00",ETAM,Fermeture Provisoire,23/06/2026 9h40,23/06/2026 10h13,9,Dimanche,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sofiane Aoues


In [291]:
df_raw.isna().sum()

Unnamed: 0       2
Unnamed: 1      54
Unnamed: 2      54
Unnamed: 3      54
Unnamed: 4      54
Unnamed: 5      54
Unnamed: 6      54
Unnamed: 7      54
Unnamed: 8      54
Unnamed: 9      73
Unnamed: 10     55
Unnamed: 11     55
Unnamed: 12     72
Unnamed: 13     87
Unnamed: 14     72
Unnamed: 15     75
Unnamed: 16    735
Unnamed: 17    735
Unnamed: 18    735
Unnamed: 19    735
Unnamed: 20    735
Unnamed: 21    735
Unnamed: 22    735
Unnamed: 23    735
Unnamed: 24    735
Unnamed: 25     66
dtype: int64

In [292]:
df_raw.shape
df_raw.columns.tolist()
df_raw.head(10)
df_raw.tail(10)
df_raw.info()

df_raw.isna().sum()

df_raw.nunique()



<class 'pandas.DataFrame'>
RangeIndex: 735 entries, 0 to 734
Data columns (total 26 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   733 non-null    str    
 1   Unnamed: 1   681 non-null    str    
 2   Unnamed: 2   681 non-null    str    
 3   Unnamed: 3   681 non-null    str    
 4   Unnamed: 4   681 non-null    str    
 5   Unnamed: 5   681 non-null    str    
 6   Unnamed: 6   681 non-null    str    
 7   Unnamed: 7   681 non-null    str    
 8   Unnamed: 8   681 non-null    str    
 9   Unnamed: 9   662 non-null    str    
 10  Unnamed: 10  680 non-null    str    
 11  Unnamed: 11  680 non-null    str    
 12  Unnamed: 12  663 non-null    str    
 13  Unnamed: 13  648 non-null    str    
 14  Unnamed: 14  663 non-null    str    
 15  Unnamed: 15  660 non-null    str    
 16  Unnamed: 16  0 non-null      float64
 17  Unnamed: 17  0 non-null      float64
 18  Unnamed: 18  0 non-null      float64
 19  Unnamed: 19  0 non-

Unnamed: 0      57
Unnamed: 1     674
Unnamed: 2      46
Unnamed: 3      89
Unnamed: 4      10
Unnamed: 5     666
Unnamed: 6     665
Unnamed: 7      68
Unnamed: 8       5
Unnamed: 9      28
Unnamed: 10     17
Unnamed: 11     18
Unnamed: 12    648
Unnamed: 13    632
Unnamed: 14     10
Unnamed: 15     14
Unnamed: 16      0
Unnamed: 17      0
Unnamed: 18      0
Unnamed: 19      0
Unnamed: 20      0
Unnamed: 21      0
Unnamed: 22      0
Unnamed: 23      0
Unnamed: 24      0
Unnamed: 25      3
dtype: int64

In [293]:
HEADER_ROW = 3

df_source = df_raw.iloc[HEADER_ROW:].copy()
df_source.columns = df_raw.iloc[HEADER_ROW]
df_source = df_source.iloc[1:].reset_index(drop=True)

df_source = df_source.dropna(axis=1, how="all")

print(f"Dimensions : {df_source.shape}")
print(df_source.columns.tolist())

Dimensions : (731, 17)
['Date du ticket', 'N° Ticket', 'Client', 'Problématique', 'Priorité', 'Date/Heure ouverture', 'Date/Heure résolution', 'Temps de résolution (h)', 'Technicien', 'Temps de résolution (min)', 'Client Normalisé', 'Problématique Normalisée', 'Date et heure ouverture', 'Date et Heure résolution', 'Heure ouverture analytique', 'Jour ouverture', 'Technicien Normalisé']


In [294]:
ticket_col = "N° Ticket"

has_ticket = (
    df_source[ticket_col]
    .notna()
    & df_source[ticket_col].astype(str).str.strip().ne("")
)

print("Lignes avec numéro de ticket :", has_ticket.sum())
print("Lignes sans numéro de ticket :", (~has_ticket).sum())

df_non_ticket = df_source.loc[~has_ticket].copy()

df_non_ticket.head(20)

Lignes avec numéro de ticket : 680
Lignes sans numéro de ticket : 51


3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien,Temps de résolution (min),Client Normalisé,Problématique Normalisée,Date et heure ouverture,Date et Heure résolution,Heure ouverture analytique,Jour ouverture,Technicien Normalisé
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,25/06/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,26/06/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,29/06/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,30/06/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,1/7/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,02/07/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,03/07/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,07/07/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,08/07/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [295]:
# Audit des lignes sans numéro de ticket

df_non_ticket = df_source.loc[~has_ticket].copy()

audit_non_ticket = pd.DataFrame({
    "Date du ticket": df_non_ticket["Date du ticket"],
    "N° Ticket": df_non_ticket["N° Ticket"],
    "Client": df_non_ticket["Client"],
    "Problématique": df_non_ticket["Problématique"],
    "Technicien": df_non_ticket["Technicien"],
})

audit_non_ticket

,Date du ticket,N° Ticket,Client,Problématique,Technicien
4,NaN,NaN,NaN,NaN,NaN
20,25/06/2026,NaN,NaN,NaN,NaN
34,26/06/2026,NaN,NaN,NaN,NaN
48,29/06/2026,NaN,NaN,NaN,NaN
62,30/06/2026,NaN,NaN,NaN,NaN
70,1/7/2026,NaN,NaN,NaN,NaN
75,02/07/2026,NaN,NaN,NaN,NaN
84,03/07/2026,NaN,NaN,NaN,NaN
108,07/07/2026,NaN,NaN,NaN,NaN
128,08/07/2026,NaN,NaN,NaN,NaN


In [296]:
# Nombre de lignes sans ticket selon leur contenu

non_empty_cells = df_non_ticket.notna().sum(axis=1)

print("Lignes sans ticket :", len(df_non_ticket))
print()
print("Répartition du nombre de cellules renseignées :")
print(non_empty_cells.value_counts().sort_index())

Lignes sans ticket : 51

Répartition du nombre de cellules renseignées :
0     1
1    49
4     1
Name: count, dtype: int64


In [297]:
# Audit des identifiants de tickets

ticket_values = (
    df_source.loc[has_ticket, "N° Ticket"]
    .astype(str)
    .str.strip()
)

print("Nombre de tickets :", len(ticket_values))
print("Nombre de tickets uniques :", ticket_values.nunique())
print("Nombre de doublons :", ticket_values.duplicated().sum())

Nombre de tickets : 680
Nombre de tickets uniques : 673
Nombre de doublons : 7


In [298]:
duplicates = ticket_values[ticket_values.duplicated(keep=False)]

print("Tickets apparaissant plusieurs fois :")
print(duplicates.value_counts())

Tickets apparaissant plusieurs fois :
N° Ticket
P260604692    2
P260700607    2
P260701439    2
P260701627    2
P260803566    2
P260803603    2
P260900721    2
Name: count, dtype: int64


In [299]:
# Audit de la période couverte

dates_ticket = pd.to_datetime(
    df_source.loc[has_ticket, "Date du ticket"],
    dayfirst=True,
    errors="coerce"
)

print("Date minimale :", dates_ticket.min())
print("Date maximale :", dates_ticket.max())
print("Dates non interprétables :", dates_ticket.isna().sum())

Date minimale : 2026-06-23 00:00:00
Date maximale : 2026-09-14 00:00:00
Dates non interprétables : 0


In [300]:
# 1 — lignes sans ticket
audit_non_ticket

# 2 — structure des lignes sans ticket
non_empty_cells.value_counts().sort_index()

# 3 — doublons tickets
print(ticket_values.nunique())
print(ticket_values.duplicated().sum())

# 4 — période
print(dates_ticket.min())
print(dates_ticket.max())
print(dates_ticket.isna().sum())

673
7
2026-06-23 00:00:00
2026-09-14 00:00:00
0


In [301]:
# Audit détaillé des doublons de numéros de ticket

duplicate_ids = (
    ticket_values[ticket_values.duplicated(keep=False)]
    .drop_duplicates()
    .tolist()
)

duplicates_detail = (
    df_source[
        df_source["N° Ticket"]
        .astype(str)
        .str.strip()
        .isin(duplicate_ids)
    ]
    [
        [
            "Date du ticket",
            "N° Ticket",
            "Client",
            "Problématique",
            "Priorité",
            "Date/Heure ouverture",
            "Date/Heure résolution",
            "Temps de résolution (h)",
            "Technicien",
        ]
    ]
    .sort_values("N° Ticket")
)

duplicates_detail

3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien
41,26/06/2026,P260604692,amplifon,ENVOI ecrant dell,Faible,26/06/2026 a 09:32,26/06/2026 a 09:37,6min,LOUBNA
78,02/07/2026,P260604692,AMPLIFON,Changement poste,basse,02/07/2026 à 13:42,02/07/2026 à 14h51,09min,LOUBNA
93,03/07/2026,P260700607,Aem Soft,Intervention J+1,faible,03/07/2026 à 13:13,03/07/2026 à 13/25,12MIN,LOUBNA
107,06/07/2026,P260700607,Aem Soft,DEMAND,Faible,06/07/2026 à 15:44,06/07/2026 à 16:01,12min,LOUBNA
138,08/07/2026,P260701439,But,Maintenance,Faible,08/07/2026 à 14:25,08/07/2026 à 14:30,00h4min,Sofiane AOUES
139,08/07/2026,P260701439,but,Maintenance,Faible,08/07/2026 à 14:30,08/07/2026 à 14:41,11min,LOUBNA
151,09/07/2026,P260701627,Shoppertrak,Maintenance,Faible,09/07/2026 à 13:45,09/07/2026 à 13:55,00h6min,Sofiane AOUES
152,09/07/2026,P260701627,Shoppertrak,Maintenance,Faible,09/07/2026 à 13:44,09/07/2026 à 13:556,13min,LOUBNA
579,21/08/2026,P260803566,Etam,Envoi Matériel,Basse,21/08/2026 à 09:00,21/08/2026 à 09:06,6min,Sofiane Aoues
581,21/08/2026,P260803566,Etam,Envoi Matériel,Basse,21/08/2026 à 09:00,21/08/2026 à 09:06,6min,Sofiane Aoues


In [302]:
# Comparaison des lignes pour chaque ticket dupliqué

for ticket_id in duplicate_ids:
    print(f"\n{'=' * 80}")
    print(f"TICKET : {ticket_id}")
    print(f"{'=' * 80}")
    
    display(
        duplicates_detail[
            duplicates_detail["N° Ticket"].astype(str).str.strip() == ticket_id
        ]
    )


TICKET : P260604692


3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien
41,26/06/2026,P260604692,amplifon,ENVOI ecrant dell,Faible,26/06/2026 a 09:32,26/06/2026 a 09:37,6min,LOUBNA
78,02/07/2026,P260604692,AMPLIFON,Changement poste,basse,02/07/2026 à 13:42,02/07/2026 à 14h51,09min,LOUBNA



TICKET : P260700607


3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien
93,03/07/2026,P260700607,Aem Soft,Intervention J+1,faible,03/07/2026 à 13:13,03/07/2026 à 13/25,12MIN,LOUBNA
107,06/07/2026,P260700607,Aem Soft,DEMAND,Faible,06/07/2026 à 15:44,06/07/2026 à 16:01,12min,LOUBNA



TICKET : P260701439


...

3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien
579,21/08/2026,P260803566,Etam,Envoi Matériel,Basse,21/08/2026 à 09:00,21/08/2026 à 09:06,6min,Sofiane Aoues
581,21/08/2026,P260803566,Etam,Envoi Matériel,Basse,21/08/2026 à 09:00,21/08/2026 à 09:06,6min,Sofiane Aoues



TICKET : P260803603


3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien
580,21/08/2026,P260803603,Etam,Swap Transporteur,Basse,21/08/2026 à 10:05,21/08/2026 à 10:09,4min,Sofiane Aoues
582,21/08/2026,P260803603,Etam,Swap Transporteur,Basse,21/08/2026 à 10:05,21/08/2026 à 10:09,4min,Sofiane Aoues



TICKET : P260900721


3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien
717,02/09/2026,P260900721,Pos Service,Installation,Basse,02/09/2026 à 16:00,02/09/2026 à 16:03,3min,Sofiane Aoues
718,02/09/2026,P260900721,Pos Service,Installation,Basse,02/09/2026 à 16:03,02/09/2026 à 16:06,3min,Sofiane Aoues


In [303]:
# Nombre de lignes enregistrées par date

tickets_par_jour = (
    df_source.loc[has_ticket]
    .assign(
        date_ticket=pd.to_datetime(
            df_source.loc[has_ticket, "Date du ticket"],
            dayfirst=True,
            errors="coerce"
        )
    )
    .groupby("date_ticket")
    .size()
    .sort_index()
)

tickets_par_jour

date_ticket
2026-06-23     5
2026-06-24    14
2026-06-25    13
2026-06-26    13
2026-06-29    13
2026-06-30     7
2026-07-01     4
2026-07-02     8
2026-07-03    13
2026-07-06    10
2026-07-07    19
2026-07-08    13
2026-07-09    14
2026-07-10    13
2026-07-13     7
2026-07-15    19
2026-07-16    14
2026-07-17    15
2026-07-20     8
2026-07-21     4
2026-07-22    10
2026-07-23    12
2026-07-24    14
2026-07-27    11
2026-07-28    14
2026-07-29    27
2026-07-30    31
2026-07-31    16
2026-08-03    21
2026-08-04    27
2026-08-05    24
2026-08-06    13
2026-08-07     7
2026-08-10    11
2026-08-11    12
2026-08-12     9
2026-08-13    14
2026-08-14    12
2026-08-17    13
2026-08-18     2
2026-08-19     5
2026-08-20     7
2026-08-21    21
2026-08-24    20
2026-08-25    21
2026-08-26    15
2026-08-27    21
2026-08-28    11
2026-08-31     9
2026-09-01     6
2026-09-02     9
2026-09-03     8
2026-09-14     1
dtype: int64

In [304]:
# Nombre cumulé de lignes avec ticket dans le temps

tickets_par_jour_cumule = tickets_par_jour.cumsum()

tickets_par_jour_cumule

date_ticket
2026-06-23      5
2026-06-24     19
2026-06-25     32
2026-06-26     45
2026-06-29     58
2026-06-30     65
2026-07-01     69
2026-07-02     77
2026-07-03     90
2026-07-06    100
2026-07-07    119
2026-07-08    132
2026-07-09    146
2026-07-10    159
2026-07-13    166
2026-07-15    185
2026-07-16    199
2026-07-17    214
2026-07-20    222
2026-07-21    226
2026-07-22    236
2026-07-23    248
2026-07-24    262
2026-07-27    273
2026-07-28    287
2026-07-29    314
2026-07-30    345
2026-07-31    361
2026-08-03    382
2026-08-04    409
2026-08-05    433
2026-08-06    446
2026-08-07    453
2026-08-10    464
2026-08-11    476
2026-08-12    485
2026-08-13    499
2026-08-14    511
2026-08-17    524
2026-08-18    526
2026-08-19    531
2026-08-20    538
2026-08-21    559
2026-08-24    579
2026-08-25    600
2026-08-26    615
2026-08-27    636
2026-08-28    647
2026-08-31    656
2026-09-01    662
2026-09-02    671
2026-09-03    679
2026-09-14    680
dtype: int64

In [305]:
# Répartition mensuelle

df_temp = df_source.loc[has_ticket].copy()

df_temp["date_ticket"] = pd.to_datetime(
    df_temp["Date du ticket"],
    dayfirst=True,
    errors="coerce"
)

print(
    df_temp["date_ticket"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

date_ticket
2026-06     65
2026-07    296
2026-08    295
2026-09     24
Freq: M, Name: count, dtype: int64


In [306]:
df_source.loc[
    df_source["Date du ticket"].astype(str).str.contains("14/09/2026", na=False),
    [
        "Date du ticket",
        "N° Ticket",
        "Client",
        "Problématique",
        "Priorité",
        "Date/Heure ouverture",
        "Date/Heure résolution",
        "Temps de résolution (h)",
        "Technicien"
    ]
]

3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien
729,14/09/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
730,14/09/2026,P260903270,Etam,Swap Transporteur,Basse,14/09/2026 à 15:32,14/09/2026 à 15:35,3min,Sofiane Aoues


In [307]:
source_cols = [
    "Date du ticket",
    "N° Ticket",
    "Client",
    "Problématique",
    "Priorité",
    "Date/Heure ouverture",
    "Date/Heure résolution",
    "Temps de résolution (h)",
    "Technicien",
]

completeness = pd.DataFrame({
    "Renseigné": df_source[source_cols].notna().sum(),
    "Manquant": df_source[source_cols].isna().sum(),
})

completeness["% renseigné"] = (
    completeness["Renseigné"] / len(df_source) * 100
).round(2)

completeness

,Renseigné,Manquant,% renseigné
3,,,
Date du ticket,730,1,99.86
N° Ticket,680,51,93.02
Client,680,51,93.02
Problématique,680,51,93.02
Priorité,680,51,93.02
Date/Heure ouverture,680,51,93.02
Date/Heure résolution,680,51,93.02
Temps de résolution (h),680,51,93.02
Technicien,680,51,93.02


## 04  Nettoyage des données

In [308]:
# Colonnes sources utilisées pour l'analyse

source_cols = [
    "Date du ticket",
    "N° Ticket",
    "Client",
    "Problématique",
    "Priorité",
    "Date/Heure ouverture",
    "Date/Heure résolution",
    "Temps de résolution (h)",
    "Technicien",
]

# Sélection des lignes correspondant réellement à des tickets
df_clean = df_source.loc[has_ticket, source_cols].copy()

# Réinitialisation de l'index après suppression des lignes de séparation
df_clean = df_clean.reset_index(drop=True)

print(f"Nombre de lignes analytiques : {len(df_clean)}")
print(f"Nombre de colonnes : {df_clean.shape[1]}")
print()
print(df_clean.head())

Nombre de lignes analytiques : 680
Nombre de colonnes : 9

3 Date du ticket   N° Ticket       Client  \
0     23/06/2026  P260604001         Etam   
1     23/06/2026  P260604023     AMPLIFON   
2     23/06/2026  P260604025     Amplifon   
3     23/06/2026  P260604110         Etam   
4     23/06/2026  P260604148  Pos Service   

3                             Problématique Priorité Date/Heure ouverture  \
0  Fermeture provisoire : demande démontage   Faible    23/06/2026 à 9h40   
1                          Changement poste    Elevé  23/06/2026 10:30:00   
2                            Envoi Matériel   Faible   23/06/2026 à 10h40   
3                            Ajout Matériel   Faible   23/06/2026 à 14h00   
4                      Maintenance matériel    Basse   23/06/2026 à 16h25   

3 Date/Heure résolution Temps de résolution (h)     Technicien  
0    23/06/2026 à 10h13                 0h19min  Sofiane AOUES  
1    23/06/2026 à 10:44                41:14:00         LOUBNA  
2    23/06/2

In [309]:
# Contrôle de complétude du dataset analytique

quality_check = pd.DataFrame({
    "Renseigné": df_clean.notna().sum(),
    "Manquant": df_clean.isna().sum(),
})

quality_check["% renseigné"] = (
    quality_check["Renseigné"] / len(df_clean) * 100
).round(2)

quality_check

,Renseigné,Manquant,% renseigné
3,,,
Date du ticket,680,0,100.0
N° Ticket,680,0,100.0
Client,680,0,100.0
Problématique,680,0,100.0
Priorité,680,0,100.0
Date/Heure ouverture,680,0,100.0
Date/Heure résolution,680,0,100.0
Temps de résolution (h),680,0,100.0
Technicien,680,0,100.0


In [310]:
df_clean["date_ticket"] = pd.to_datetime(
    df_clean["Date du ticket"],
    dayfirst=True,
    errors="coerce"
)

In [311]:
print("Dates non interprétables :", df_clean["date_ticket"].isna().sum())
print("Date minimale :", df_clean["date_ticket"].min())
print("Date maximale :", df_clean["date_ticket"].max())

Dates non interprétables : 0
Date minimale : 2026-06-23 00:00:00
Date maximale : 2026-09-14 00:00:00


In [312]:
import re

def parse_resolution_minutes(value):
    """
    Convertit les différents formats présents dans
    'Temps de résolution (h)' en minutes.

    Formats gérés :
    - 3min
    - 10MIN
    - 0h19min
    - 00h5min
    - 41:14:00  -> 41 min 14 sec
    - 28:00:00  -> 28 min
    - 16:00     -> 16 min
    - 15:18min  -> 15 min 18 sec
    - 14mi06    -> 14 min 06 sec
    - 10min 41  -> 10 min 41 sec
    - 25min10   -> 25 min 10 sec
    - 9mi       -> 9 min
    """

    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower()

    # --------------------------------------------------
    # 1. Format : XhYmin
    # --------------------------------------------------
    match = re.fullmatch(r"(\d+)\s*h\s*(\d+)\s*min", s)

    if match:
        hours = int(match.group(1))
        minutes = int(match.group(2))
        return hours * 60 + minutes

    # --------------------------------------------------
    # 2. Format : Xmin
    # --------------------------------------------------
    match = re.fullmatch(r"(\d+)\s*min", s)

    if match:
        return float(match.group(1))

    # --------------------------------------------------
    # 3. Format : Xmi
    # --------------------------------------------------
    match = re.fullmatch(r"(\d+)\s*mi", s)

    if match:
        return float(match.group(1))

    # --------------------------------------------------
    # 4. Format : XminYY
    #    ex. 10min 41 / 25min10
    # --------------------------------------------------
    match = re.fullmatch(r"(\d+)\s*min\s*(\d{1,2})", s)

    if match:
        minutes = int(match.group(1))
        seconds = int(match.group(2))
        return minutes + seconds / 60

    # --------------------------------------------------
    # 5. Format : XmiYY
    #    ex. 14mi06
    # --------------------------------------------------
    match = re.fullmatch(r"(\d+)\s*mi\s*(\d{1,2})", s)

    if match:
        minutes = int(match.group(1))
        seconds = int(match.group(2))
        return minutes + seconds / 60

    # --------------------------------------------------
    # 6. Format : MM:SSmin
    #    ex. 15:18min
    # --------------------------------------------------
    match = re.fullmatch(r"(\d+):(\d{1,2})\s*min", s)

    if match:
        minutes = int(match.group(1))
        seconds = int(match.group(2))
        return minutes + seconds / 60

    # --------------------------------------------------
    # 7. Format : MM:SS:00
    #    ex. 41:14:00
    #
    #    Dans notre fichier, le premier nombre correspond
    #    aux minutes et le second aux secondes.
    # --------------------------------------------------
    match = re.fullmatch(r"(\d+):(\d{2}):(\d{2})", s)

    if match:
        minutes = int(match.group(1))
        seconds = int(match.group(2))
        return minutes + seconds / 60

    # --------------------------------------------------
    # 8. Format : MM:SS
    # --------------------------------------------------
    match = re.fullmatch(r"(\d+):(\d{2})", s)

    if match:
        minutes = int(match.group(1))
        seconds = int(match.group(2))
        return minutes + seconds / 60

    return np.nan

In [313]:
df_clean["resolution_minutes"] = (
    df_clean["Temps de résolution (h)"]
    .apply(parse_resolution_minutes)
)

In [314]:
print(
    df_clean[
        df_clean["Temps de résolution (h)"].isin([
            "41:14:00",
            "28:00:00",
            "15:18min",
            "08:18min",
            "14mi06",
            "10min 41",
            "25min10",
            "9mi"
        ])
    ][[
        "N° Ticket",
        "Temps de résolution (h)",
        "resolution_minutes"
    ]]
)

3     N° Ticket Temps de résolution (h)  resolution_minutes
1    P260604023                41:14:00           41.233333
30   P260604614                15:18min           15.300000
37   P260604679                08:18min            8.300000
50   P260605043                  14mi06           14.100000
51   P260605055                10min 41           10.683333
52   P260605066                 25min10           25.166667
130  P260701437                     9mi            9.000000


In [315]:
print(
    "\nValeurs non interprétables :",
    df_clean["resolution_minutes"].isna().sum()
)

print("\nStatistiques :")
print(
    df_clean["resolution_minutes"].describe()
)


Valeurs non interprétables : 0

Statistiques :
count    680.000000
mean       4.873211
std        4.116082
min        1.000000
25%        3.000000
50%        4.000000
75%        5.000000
max       41.233333
Name: resolution_minutes, dtype: float64


In [316]:
df_clean["technicien"] = (
    df_clean["Technicien"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df_clean["technicien"] = (
    df_clean["technicien"]
    .str.title()
)

df_clean["technicien"].value_counts()

technicien
Sofiane Aoues    601
Loubna            79
Name: count, dtype: int64

In [317]:
priority_mapping = {
    "faible": "Basse",
    "basse": "Basse",
    "moyenne": "Moyenne",
    "élevé": "Haute",
    "eleve": "Haute",
    "haute": "Haute",
}

df_clean["priorite"] = (
    df_clean["Priorité"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(priority_mapping)
)

print(df_clean["priorite"].value_counts(dropna=False))

priorite
Basse    675
NaN        5
Name: count, dtype: int64


In [318]:
print(
    "Priorités non reconnues :",
    df_clean["priorite"].isna().sum()
)

if df_clean["priorite"].isna().any():
    print(
        df_clean.loc[
            df_clean["priorite"].isna(),
            "Priorité"
        ].unique()
    )

Priorités non reconnues : 5
<StringArray>
['Elevé', 'Urgent', 'MOYEN', 'Moyen']
Length: 4, dtype: str


In [319]:
df_clean["annee"] = df_clean["date_ticket"].dt.year
df_clean["mois"] = df_clean["date_ticket"].dt.month
df_clean["mois_nom"] = df_clean["date_ticket"].dt.strftime("%B")
df_clean["semaine"] = df_clean["date_ticket"].dt.isocalendar().week
df_clean["jour_semaine"] = df_clean["date_ticket"].dt.day_name()

In [320]:
df_clean["technicien"].value_counts()

technicien
Sofiane Aoues    601
Loubna            79
Name: count, dtype: int64

In [321]:
df_clean["priorite"].value_counts(dropna=False)

priorite
Basse    675
NaN        5
Name: count, dtype: int64

In [322]:
priority_unknown = df_clean[
    df_clean["priorite"].isna()
][[
    "N° Ticket",
    "Date du ticket",
    "Client",
    "Problématique",
    "Priorité",
    "Technicien"
]]

priority_unknown

3,N° Ticket,Date du ticket,Client,Problématique,Priorité,Technicien
1,P260604023,23/06/2026,AMPLIFON,Changement poste,Elevé,LOUBNA
9,P260604283,24/06/2026,etam,Maintenance v400,Urgent,LOUBNA
56,P260605122,29/06/2026,etam,remplacement tpe,MOYEN,LOUBNA
277,P260704380,28/07/2026,Etam,Intervention Site,Moyen,Sofiane Aoues
511,P260802752,17/08/2026,Pos Service,Maintenance,Urgent,Sofiane Aoues


In [323]:
priority_mapping = {
    "faible": "Basse",
    "basse": "Basse",

    "moyenne": "Moyenne",
    "moyen": "Moyenne",

    "élevé": "Haute",
    "eleve": "Haute",
    "elevé": "Haute",
    "haute": "Haute",

    "urgent": "Urgente",
}

In [324]:
df_clean["priorite"] = (
    df_clean["Priorité"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(priority_mapping)
)

In [325]:
print(df_clean["priorite"].value_counts(dropna=False))

priorite
Basse      675
Urgente      2
Moyenne      2
Haute        1
Name: count, dtype: int64


In [326]:
print(
    "Priorités non reconnues :",
    df_clean["priorite"].isna().sum()
)

Priorités non reconnues : 0


In [327]:
df_clean["Date/Heure ouverture"].value_counts().head(30)

Date/Heure ouverture
24/06/2026 a 09:07     3
25/06/2026 à 11h10     2
29/06/2026 à 10h30     2
29/06/2026 à 11h10     2
29/06/2026 à 13h15     2
02/07/2026 à 14h25     2
03/07/2026 à 10:55     2
06/07/2026 à 10:00     2
07/07/2026 à 14:30     2
08/07/2026 à 10:50     2
16/07/2026 à 14h25     2
04/08/2026 à 09:52     2
 21/08/2026 à 09:00    2
 21/08/2026 à 10:05    2
23/06/2026 à 9h40      1
23/06/2026 10:30:00    1
23/06/2026 à 10h40     1
23/06/2026 à 14h00     1
23/06/2026 à 16h25     1
24/06/2026 à 10h05     1
24/06/2026 à 10h35     1
24/06/2026 a 13:08     1
24/06/2026 à 13h05     1
24/06/2026 à 13h15     1
24/06/2026 à 13h20     1
24/06/2026 à 14h45     1
24/06/2026 à 15h50     1
24/06/2026 à 15h11     1
24/06/2026 à 16h30     1
24/06/2026 à 16h40     1
Name: count, dtype: int64

In [328]:
print(
    df_clean["Date/Heure ouverture"]
    .astype(str)
    .str.extract(r"(\d{1,2}h\d{1,2}|\d{1,2}:\d{2}:\d{2}|\d{1,2}:\d{2})", expand=False)
    .value_counts()
    .head(30)
)

Date/Heure ouverture
10:00    12
09:00    10
14:00     9
14:20     8
14:05     8
15:30     7
11:00     7
14:30     7
11:30     7
09:07     6
11h10     6
11:40     6
09:40     6
10:50     6
10:55     6
14:50     6
09:10     6
09:25     6
15:00     6
14:25     6
11:10     6
10:05     6
10:42     5
15:45     5
15:50     5
09:30     5
15:10     5
15:40     5
15:20     5
14:55     5
Name: count, dtype: int64


In [329]:
df_clean["ouverture_normalisee"] = (
    df_clean["Date/Heure ouverture"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" à ", " ", regex=False)
    .str.replace(" a ", " ", regex=False)
    .str.replace("h", ":", regex=False)
)

In [330]:
df_clean["date_heure_ouverture"] = pd.to_datetime(
    df_clean["ouverture_normalisee"],
    dayfirst=True,
    errors="coerce"
)

In [331]:
print(
    "Dates/heures d'ouverture non interprétables :",
    df_clean["date_heure_ouverture"].isna().sum()
)

print(
    "Ouverture minimale :",
    df_clean["date_heure_ouverture"].min()
)

print(
    "Ouverture maximale :",
    df_clean["date_heure_ouverture"].max()
)

Dates/heures d'ouverture non interprétables : 5
Ouverture minimale : 2026-06-23 09:40:00
Ouverture maximale : 2026-09-14 15:32:00


In [332]:
df_clean.loc[
    df_clean["date_heure_ouverture"].isna(),
    [
        "N° Ticket",
        "Date/Heure ouverture",
        "ouverture_normalisee"
    ]
]

3,N° Ticket,Date/Heure ouverture,ouverture_normalisee
1,P260604023,23/06/2026 10:30:00,23/06/2026 10:30:00
57,P260605172,29/06/2026 à 1643,29/06/2026 1643
77,P260700511,03/07/2026 à 09:30:00,03/07/2026 09:30:00
110,P260701201,07/07/2026 à 14/10,07/07/2026 14/10
194,P260702686,16/07/2026 à 14/43,16/07/2026 14/43


In [333]:
df_clean["heure_ouverture"] = (
    df_clean["date_heure_ouverture"].dt.hour
)

df_clean["minute_ouverture"] = (
    df_clean["date_heure_ouverture"].dt.minute
)

df_clean["jour_ouverture"] = (
    df_clean["date_heure_ouverture"].dt.day_name()
)

In [334]:
import re
import pandas as pd
import numpy as np


def normalize_opening_datetime(value):
    """Normalise les différents formats de date/heure d'ouverture."""

    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower()

    # Séparateurs textuels
    s = s.replace(" à ", " ")
    s = s.replace(" a ", " ")

    # Heure avec 'h' :
    # 9h50 -> 9:50
    # 14h30 -> 14:30
    s = re.sub(r"(\d{1,2})h(\d{2})", r"\1:\2", s)

    # Heure écrite HHMM :
    # 1643 -> 16:43
    match = re.fullmatch(
        r"(\d{1,2}/\d{1,2}/\d{4})\s+(\d{4})",
        s
    )

    if match:
        date_part, time_part = match.groups()
        s = f"{date_part} {time_part[:2]}:{time_part[2:]}"

    # Heure écrite HH/MM :
    # 14/10 -> 14:10
    match = re.fullmatch(
        r"(\d{1,2}/\d{1,2}/\d{4})\s+(\d{1,2})/(\d{2})",
        s
    )

    if match:
        date_part, hour, minute = match.groups()
        s = f"{date_part} {hour}:{minute}"

    return s


def parse_opening_datetime(value):
    """Convertit une chaîne normalisée en datetime."""

    if pd.isna(value):
        return pd.NaT

    s = str(value).strip()

    # Date + heure avec ou sans secondes
    match = re.fullmatch(
        r"(\d{1,2}/\d{1,2}/\d{4})\s+"
        r"(\d{1,2}):(\d{2})"
        r"(?::(\d{2}))?",
        s
    )

    if not match:
        return pd.NaT

    date_part, hour, minute, second = match.groups()

    if second is None:
        second = "00"

    datetime_string = (
        f"{date_part} {hour}:{minute}:{second}"
    )

    return pd.to_datetime(
        datetime_string,
        format="%d/%m/%Y %H:%M:%S",
        errors="coerce"
    )


# 1. Normalisation
df_clean["ouverture_normalisee"] = (
    df_clean["Date/Heure ouverture"]
    .apply(normalize_opening_datetime)
)

# 2. Conversion datetime
df_clean["date_heure_ouverture"] = (
    df_clean["ouverture_normalisee"]
    .apply(parse_opening_datetime)
)

In [335]:
print(
    "Dates/heures d'ouverture non interprétables :",
    df_clean["date_heure_ouverture"].isna().sum()
)

print(
    "Ouverture minimale :",
    df_clean["date_heure_ouverture"].min()
)

print(
    "Ouverture maximale :",
    df_clean["date_heure_ouverture"].max()
)

Dates/heures d'ouverture non interprétables : 0
Ouverture minimale : 2026-06-23 09:40:00
Ouverture maximale : 2026-09-14 15:32:00


In [336]:
df_clean.loc[
    df_clean["N° Ticket"].isin([
        "P260700078",
        "P260700192",
        "P260700196"
    ]),
    [
        "N° Ticket",
        "Date/Heure ouverture",
        "ouverture_normalisee",
        "date_heure_ouverture"
    ]
]

3,N° Ticket,Date/Heure ouverture,ouverture_normalisee,date_heure_ouverture
65,P260700078,1/07/2026 à 9h50,1/07/2026 9:50,2026-07-01 09:50:00
66,P260700192,1/07/2026 à 14h30,1/07/2026 14:30,2026-07-01 14:30:00
68,P260700196,1/07/2026 à 14h42,1/07/2026 14:42,2026-07-01 14:42:00


In [337]:
df_clean["heure_ouverture"] = (
    df_clean["date_heure_ouverture"].dt.hour
)

df_clean["minute_ouverture"] = (
    df_clean["date_heure_ouverture"].dt.minute
)

df_clean["jour_ouverture"] = (
    df_clean["date_heure_ouverture"].dt.day_name()
)

In [338]:
df_clean[
    [
        "date_heure_ouverture",
        "heure_ouverture",
        "minute_ouverture",
        "jour_ouverture"
    ]
].head(10)

3,date_heure_ouverture,heure_ouverture,minute_ouverture,jour_ouverture
0,2026-06-23 09:40:00,9,40,Tuesday
1,2026-06-23 10:30:00,10,30,Tuesday
2,2026-06-23 10:40:00,10,40,Tuesday
3,2026-06-23 14:00:00,14,0,Tuesday
4,2026-06-23 16:25:00,16,25,Tuesday
5,2026-06-24 10:05:00,10,5,Wednesday
6,2026-06-24 10:35:00,10,35,Wednesday
7,2026-06-24 09:07:00,9,7,Wednesday
8,2026-06-24 09:07:00,9,7,Wednesday
9,2026-06-24 09:07:00,9,7,Wednesday


In [339]:
jours_fr = {
    "Monday": "Lundi",
    "Tuesday": "Mardi",
    "Wednesday": "Mercredi",
    "Thursday": "Jeudi",
    "Friday": "Vendredi",
    "Saturday": "Samedi",
    "Sunday": "Dimanche",
}

df_clean["jour_ouverture_fr"] = (
    df_clean["jour_ouverture"].map(jours_fr)
)

In [340]:
print("Heures d'ouverture :")
print(
    df_clean["heure_ouverture"]
    .value_counts()
    .sort_index()
)

print("\nJours d'ouverture :")
print(
    df_clean["jour_ouverture_fr"]
    .value_counts()
)

Heures d'ouverture :
heure_ouverture
8       3
9     112
10    139
11     88
12     26
13     43
14    126
15    106
16     37
Name: count, dtype: int64

Jours d'ouverture :
jour_ouverture_fr
Jeudi       156
Mercredi    149
Vendredi    134
Lundi       125
Mardi       116
Name: count, dtype: int64


In [341]:
heure_counts = (
    df_clean["heure_ouverture"]
    .value_counts()
    .sort_index()
    .reset_index()
)

heure_counts.columns = ["heure", "tickets"]

heure_counts

,heure,tickets
0,8,3
1,9,112
2,10,139
3,11,88
4,12,26
5,13,43
6,14,126
7,15,106
8,16,37


In [342]:
fig = px.bar(
    heure_counts,
    x="heure",
    y="tickets",
    title="Volume des tickets par heure d'ouverture",
    labels={
        "heure": "Heure d'ouverture",
        "tickets": "Nombre de tickets"
    }
)

fig.update_layout(
    template="plotly_white",
    height=450
)

fig.show()

In [343]:
df_clean["Date/Heure résolution"].drop_duplicates().head(30)

0     23/06/2026 à 10h13
1     23/06/2026 à 10:44
2     23/06/2026 à 10h48
3     23/06/2026 à 14h12
4     23/06/2026 à 16h40
5     24/06/2026 à 10h15
6     24/06/2026 à 10h45
7     24/06/2026 a 09:42
8     24/06/2026 a 09:07
10    24/06/2026 a 13:33
11    24/06/2026 à 13h10
12    24/06/2026 à 13h20
13    24/06/2026 à 13h30
14    24/06/2026 à 14h50
15    24/06/2026 à 16h00
16    24/06/2026 à 16h01
17    24/06/2026 à 16h40
18    24/06/2026 à 16h50
19     25/06/2026 à 9h40
20    25/06/2026 à 10h20
21    25/06/2026 à 10h25
22    25/06/2026 à 10h10
23    25/06/2026 à 11h10
25    25/06/2026 à 13h30
26    25/06/2026 à 14:45
27    25/06/2026 à 14h10
28    25/06/2026 à 15h20
29    25/06/2026 à 16:15
30    25/06/2026 à 16:40
31    25/06/2026 à 16h00
Name: Date/Heure résolution, dtype: str

In [344]:
df_clean["Date/Heure résolution"].astype(str).str.extract(
    r"(\d{1,2}/\d{1,2}/\d{4})"
)[0].value_counts().head()

0
30/07/2026    31
29/07/2026    27
04/08/2026    27
05/08/2026    24
21/08/2026    21
Name: count, dtype: int64

In [345]:
df_clean["Date/Heure résolution"].astype(str).str.extract(
    r"\s(?:à|a)?\s*(.*)$"
)[0].value_counts().head(30)

0
09:35    6
14:10    6
13h30    5
09:10    5
10:20    5
14h50    4
11h10    4
11h20    4
14:06    4
10:00    4
10:55    4
11:00    4
14:30    4
09:30    4
09:40    4
10:35    4
09:20    4
10:03    4
10:40    4
14:18    4
14:48    4
10:10    4
11:20    4
10:44    3
09:07    3
13h20    3
16h00    3
10:57    3
14h30    3
15:45    3
Name: count, dtype: int64

In [346]:
import re
import pandas as pd
import numpy as np


def normalize_resolution_datetime(value):
    """
    Normalise les différents formats de date/heure de résolution.
    """

    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower()

    # Normalisation du séparateur 'à' ou 'a',
    # y compris lorsque les espaces sont absents.
    s = re.sub(r"\s*[àa]\s*", " ", s)

    # Séparation entre la date et l'heure
    match = re.fullmatch(
        r"(\d{1,2}/\d{1,2}/\d{4})\s*(.*)",
        s
    )

    if not match:
        return s

    date_part, time_part = match.groups()
    time_part = time_part.strip()

    # Cas d'une date seule : on conserve uniquement la date
    if not time_part:
        return date_part

    # Exemple : 9h40 -> 9:40
    time_part = re.sub(
        r"^(\d{1,2})h(\d{2})$",
        r"\1:\2",
        time_part
    )

    # Exemple : 09/47 -> 09:47
    # Cette conversion est appliquée uniquement à la partie heure.
    time_part = re.sub(
        r"^(\d{1,2})/(\d{2})$",
        r"\1:\2",
        time_part
    )

    return f"{date_part} {time_part}"


def parse_resolution_datetime(value):
    """
    Convertit une date/heure normalisée en objet datetime.
    Les formats invalides sont convertis en NaT.
    """

    if pd.isna(value):
        return pd.NaT

    s = str(value).strip()

    match = re.fullmatch(
        r"(\d{1,2}/\d{1,2}/\d{4})\s+"
        r"(\d{1,2}):(\d{2})"
        r"(?::(\d{2}))?",
        s
    )

    if not match:
        return pd.NaT

    date_part, hour, minute, second = match.groups()

    if second is None:
        second = "00"

    return pd.to_datetime(
        f"{date_part} {hour}:{minute}:{second}",
        format="%d/%m/%Y %H:%M:%S",
        errors="coerce"
    )

In [347]:
df_clean["resolution_normalisee"] = (
    df_clean["Date/Heure résolution"]
    .apply(normalize_resolution_datetime)
)

df_clean["date_heure_resolution"] = (
    df_clean["resolution_normalisee"]
    .apply(parse_resolution_datetime)
)

In [348]:
nb_non_interpretes = df_clean["date_heure_resolution"].isna().sum()

print(
    "Dates/heures de résolution non interprétables :",
    nb_non_interpretes
)

Dates/heures de résolution non interprétables : 1


In [349]:
df_clean.loc[
    df_clean["date_heure_resolution"].isna(),
    [
        "N° Ticket",
        "Date/Heure résolution",
        "resolution_normalisee"
    ]
]

3,N° Ticket,Date/Heure résolution,resolution_normalisee
141,P260701627,09/07/2026 à 13:556,09/07/2026 13:556


In [350]:
df_clean["resolution_datetime_valide"] = (
    df_clean["date_heure_resolution"].notna()
)

In [351]:
print(
    "Dates/heures de résolution interprétables :",
    df_clean["date_heure_resolution"].notna().sum()
)

print(
    "Dates/heures de résolution non interprétables :",
    df_clean["date_heure_resolution"].isna().sum()
)

Dates/heures de résolution interprétables : 679
Dates/heures de résolution non interprétables : 1


In [352]:
anomalies_resolution = df_clean.loc[
    df_clean["date_heure_resolution"].isna(),
    [
        "N° Ticket",
        "Date/Heure ouverture",
        "Date/Heure résolution",
        "Temps de résolution (h)"
    ]
]

anomalies_resolution

3,N° Ticket,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h)
141,P260701627,09/07/2026 à 13:44,09/07/2026 à 13:556,13min


In [353]:
df_clean.loc[
    df_clean["N° Ticket"] == "P260701627",
    "Date/Heure résolution"
] = "09/07/2026 à 13:56"

In [354]:
df_clean["resolution_normalisee"] = (
    df_clean["Date/Heure résolution"]
    .apply(normalize_resolution_datetime)
)

df_clean["date_heure_resolution"] = (
    df_clean["resolution_normalisee"]
    .apply(parse_resolution_datetime)
)

In [355]:
df_clean.loc[
    df_clean["N° Ticket"] == "P260701627",
    [
        "N° Ticket",
        "Date/Heure résolution",
        "resolution_normalisee",
        "date_heure_resolution"
    ]
]

3,N° Ticket,Date/Heure résolution,resolution_normalisee,date_heure_resolution
140,P260701627,09/07/2026 à 13:56,09/07/2026 13:56,2026-07-09 13:56:00
141,P260701627,09/07/2026 à 13:56,09/07/2026 13:56,2026-07-09 13:56:00


In [356]:
print(
    "Dates/heures de résolution non interprétables :",
    df_clean["date_heure_resolution"].isna().sum()
)

Dates/heures de résolution non interprétables : 0


In [357]:
valid_chrono = (
    df_clean["date_heure_ouverture"].notna()
    & df_clean["date_heure_resolution"].notna()
)

anomalies_chronologiques = df_clean.loc[
    valid_chrono
    & (
        df_clean["date_heure_resolution"]
        < df_clean["date_heure_ouverture"]
    ),
    [
        "N° Ticket",
        "Date/Heure ouverture",
        "Date/Heure résolution",
        "date_heure_ouverture",
        "date_heure_resolution",
        "Temps de résolution (h)"
    ]
]

print(
    "Nombre d'anomalies chronologiques :",
    len(anomalies_chronologiques)
)

anomalies_chronologiques

Nombre d'anomalies chronologiques : 16


3,N° Ticket,Date/Heure ouverture,Date/Heure résolution,date_heure_ouverture,date_heure_resolution,Temps de résolution (h)
62,P260605452,30/06/2026 à 15:30,30/06/2026 à 14:49,2026-06-30 15:30:00,2026-06-30 14:49:00,19min
105,P260701126,07/07/2026 à 11:11,07/07/2026 à 11:10,2026-07-07 11:11:00,2026-07-07 11:10:00,10MIN
111,P260701208,07/07/2026 à 14:30,07/07/2026 à 13:37,2026-07-07 14:30:00,2026-07-07 13:37:00,00h6min
159,P260702003,13/07/2026 à 8h40,1/07/2026 à 08h50,2026-07-13 08:40:00,2026-07-01 08:50:00,0h5min
160,P260702032,13/07/2026 à 10h15,1/07/2026 à 10h20,2026-07-13 10:15:00,2026-07-01 10:20:00,0h5min
161,P260702040,13/07/2026 à 10h25,1/07/2026 à 10h30,2026-07-13 10:25:00,2026-07-01 10:30:00,0h3min
200,P260702791,17/07/2026 à 10:18,17/07/2026 à 09:27,2026-07-17 10:18:00,2026-07-17 09:27:00,9min
274,P260704344,28/07/2026 à 09:36,28/07/2026 à 09:35,2026-07-28 09:36:00,2026-07-28 09:35:00,2min
373,P260800181,03/08/2026 à 11:27,31/07/2026 à 11:30,2026-08-03 11:27:00,2026-07-31 11:30:00,3min
447,P260801328,07/08/2026 à 09:53,06/08/2026 à 09:55,2026-08-07 09:53:00,2026-08-06 09:55:00,2min


In [358]:
df_clean["duree_calculee_minutes"] = (
    df_clean["date_heure_resolution"]
    - df_clean["date_heure_ouverture"]
).dt.total_seconds() / 60

In [359]:
df_clean["ecart_duree_minutes"] = (
    df_clean["duree_calculee_minutes"]
    - df_clean["resolution_minutes"]
)

In [360]:
ecarts_duree = df_clean.loc[
    df_clean["ecart_duree_minutes"].abs() > 2,
    [
        "N° Ticket",
        "Date/Heure ouverture",
        "Date/Heure résolution",
        "Temps de résolution (h)",
        "resolution_minutes",
        "duree_calculee_minutes",
        "ecart_duree_minutes"
    ]
].sort_values("ecart_duree_minutes")

ecarts_duree

3,N° Ticket,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),resolution_minutes,duree_calculee_minutes,ecart_duree_minutes
160,P260702032,13/07/2026 à 10h15,1/07/2026 à 10h20,0h5min,5.000000,-17275.0,-17280.000000
161,P260702040,13/07/2026 à 10h25,1/07/2026 à 10h30,0h3min,3.000000,-17275.0,-17278.000000
159,P260702003,13/07/2026 à 8h40,1/07/2026 à 08h50,0h5min,5.000000,-17270.0,-17275.000000
373,P260800181,03/08/2026 à 11:27,31/07/2026 à 11:30,3min,3.000000,-4317.0,-4320.000000
451,P260801425,07/08/2026 à 14:22,06/08/2026 à 14:25,3min,3.000000,-1437.0,-1440.000000
447,P260801328,07/08/2026 à 09:53,06/08/2026 à 09:55,2min,2.000000,-1438.0,-1440.000000
449,P260801374,07/08/2026 à 11:25,06/08/2026 à 11:27,2min,2.000000,-1438.0,-1440.000000
448,P260801356,07/08/2026 à 10:33,06/08/2026 à 10:36,3min,3.000000,-1437.0,-1440.000000
450,P260801378,07/08/2026 à 11:35,06/08/2026 à 11:38,3min,3.000000,-1437.0,-1440.000000
452,P260801450,07/08/2026 à 15:50,06/08/2026 à 15:58,8min,8.000000,-1432.0,-1440.000000


In [361]:
df_clean["coherence_duree"] = np.select(
    [
        df_clean["ecart_duree_minutes"].abs() <= 2,
        df_clean["ecart_duree_minutes"].abs() > 2
    ],
    [
        "Cohérente",
        "À vérifier"
    ],
    default="Non évaluable"
)

In [362]:
df_clean["coherence_duree"].value_counts(dropna=False)

coherence_duree
Cohérente     598
À vérifier     82
Name: count, dtype: int64

In [363]:
anomalies_chronologiques

3,N° Ticket,Date/Heure ouverture,Date/Heure résolution,date_heure_ouverture,date_heure_resolution,Temps de résolution (h)
62,P260605452,30/06/2026 à 15:30,30/06/2026 à 14:49,2026-06-30 15:30:00,2026-06-30 14:49:00,19min
105,P260701126,07/07/2026 à 11:11,07/07/2026 à 11:10,2026-07-07 11:11:00,2026-07-07 11:10:00,10MIN
111,P260701208,07/07/2026 à 14:30,07/07/2026 à 13:37,2026-07-07 14:30:00,2026-07-07 13:37:00,00h6min
159,P260702003,13/07/2026 à 8h40,1/07/2026 à 08h50,2026-07-13 08:40:00,2026-07-01 08:50:00,0h5min
160,P260702032,13/07/2026 à 10h15,1/07/2026 à 10h20,2026-07-13 10:15:00,2026-07-01 10:20:00,0h5min
161,P260702040,13/07/2026 à 10h25,1/07/2026 à 10h30,2026-07-13 10:25:00,2026-07-01 10:30:00,0h3min
200,P260702791,17/07/2026 à 10:18,17/07/2026 à 09:27,2026-07-17 10:18:00,2026-07-17 09:27:00,9min
274,P260704344,28/07/2026 à 09:36,28/07/2026 à 09:35,2026-07-28 09:36:00,2026-07-28 09:35:00,2min
373,P260800181,03/08/2026 à 11:27,31/07/2026 à 11:30,2026-08-03 11:27:00,2026-07-31 11:30:00,3min
447,P260801328,07/08/2026 à 09:53,06/08/2026 à 09:55,2026-08-07 09:53:00,2026-08-06 09:55:00,2min


In [364]:
taux_coherence = (
    df_clean["coherence_duree"]
    .eq("Cohérente")
    .mean()
    * 100
)

print(f"Taux de cohérence des durées : {taux_coherence:.1f} %")

Taux de cohérence des durées : 87.9 %


In [365]:
synthese_qualite = pd.DataFrame({
    "Indicateur": [
        "Nombre de lignes analytiques",
        "Durées cohérentes",
        "Durées à vérifier",
        "Anomalies chronologiques"
    ],
    "Valeur": [
        len(df_clean),
        (df_clean["coherence_duree"] == "Cohérente").sum(),
        (df_clean["coherence_duree"] == "À vérifier").sum(),
        len(anomalies_chronologiques)
    ]
})

synthese_qualite

,Indicateur,Valeur
0,Nombre de lignes analytiques,680
1,Durées cohérentes,598
2,Durées à vérifier,82
3,Anomalies chronologiques,16


In [366]:
colonnes_normalisees = [
    "Client Normalisé",
    "Problématique Normalisée"
]

df_normalisations = df_source.loc[
    has_ticket,
    colonnes_normalisees
].reset_index(drop=True)

df_clean[colonnes_normalisees] = df_normalisations

In [367]:
print(df_clean[
    [
        "Client",
        "Client Normalisé",
        "Problématique",
        "Problématique Normalisée"
    ]
].head(10))

3       Client Client Normalisé                             Problématique  \
0         Etam             ETAM  Fermeture provisoire : demande démontage   
1     AMPLIFON         AMPLIFON                          Changement poste   
2     Amplifon         AMPLIFON                            Envoi Matériel   
3         Etam             ETAM                            Ajout Matériel   
4  Pos Service      POS SERVICE                      Maintenance matériel   
5         Etam             ETAM                      Fermeture définitive   
6  Pos Service      POS SERVICE                     Installation matériel   
7        adopt            ADOPT                      INSTALLATIONS ITALIE   
8        adopt            ADOPT                      INSTALLATIONS france   
9         etam             ETAM                          Maintenance v400   

3 Problématique Normalisée  
0     Fermeture Provisoire  
1           Envoi Matériel  
2           Envoi Matériel  
3           Ajout Matériel  
4      

In [368]:
print(
    df_clean[
        [
            "Client Normalisé",
            "Problématique Normalisée"
        ]
    ].isna().sum()
)

3
Client Normalisé            1
Problématique Normalisée    1
dtype: int64


In [369]:
df_clean = df_clean.rename(
    columns={
        "Client Normalisé": "client",
        "Problématique Normalisée": "problematique"
    }
)

In [370]:
df_clean["client"]
df_clean["problematique"]

0      Fermeture Provisoire
1            Envoi Matériel
2            Envoi Matériel
3            Ajout Matériel
4               Maintenance
               ...         
675            Installation
676             Maintenance
677            Installation
678            Installation
679                     NaN
Name: problematique, Length: 680, dtype: str

In [371]:
df_clean.loc[
    df_clean["client"].isna()
    | df_clean["problematique"].isna(),
    [
        "N° Ticket",
        "Client",
        "client",
        "Problématique",
        "problematique"
    ]
]

3,N° Ticket,Client,client,Problématique,problematique
679,P260903270,Etam,NaN,Swap Transporteur,NaN


In [372]:
df_clean.loc[
    ~df_clean["N° Ticket"].astype(str).str.match(r"^P\d+$"),
    [
        "N° Ticket",
        "Date du ticket",
        "Client",
        "Problématique",
        "client",
        "problematique"
    ]
]

3,N° Ticket,Date du ticket,Client,Problématique,client,problematique
53,p260605090,29/06/2026,etam,remplacement tpe,ETAM,Remplacement Matériel
131,P260701468.,08/07/2026,Shoppertrak,Maintenance,SHOPPERTRAK,Maintenance
149,: P260701753,10/07/2026,Etam,Ajout matériel,ETAM,Ajout Matériel
169,15/07/2026,15/07/2026,ADOPT,MAINTENANCE,ADOPT,Maintenance
184,P260702427,15/07/2026,ADOPT,Maintenance,ADOPT,Maintenance
190,260702658,16/07/2026,Axe E-Sante,remplacement,AXE E-SANTE,Remplacement Matériel
211,°P260702904,17/07/2026,Etam,envoi ipad,ETAM,Envoi Matériel
268,P260704258,27/07/2026,Amplifon,Commande PC,AMPLIFON,Envoi Matériel
321,P260704761,30/07/2026,Centric Action,Handover,CENTRIC ACTION,Handover
454,P260801609,10/08/2026,Etam,Envoi Matériel,ETAM,Envoi Matériel


In [373]:
df_source.loc[
    df_source["N° Ticket"].astype(str).str.contains(
        "P260903270",
        case=False,
        na=False
    )
].T

,730
3,
Date du ticket,14/09/2026
N° Ticket,P260903270
Client,Etam
Problématique,Swap Transporteur
Priorité,Basse
Date/Heure ouverture,14/09/2026 à 15:32
Date/Heure résolution,14/09/2026 à 15:35
Temps de résolution (h),3min
Technicien,Sofiane Aoues


In [374]:
df_source.loc[
    df_source["N° Ticket"].astype(str).str.contains(
        "P260903270",
        case=False,
        na=False
    ),
    [
        "N° Ticket",
        "Client",
        "Problématique",
        "Client Normalisé",
        "Problématique Normalisée",
        "Temps de résolution (h)",
        "Technicien"
    ]
]

3,N° Ticket,Client,Problématique,Client Normalisé,Problématique Normalisée,Temps de résolution (h),Technicien
730,P260903270,Etam,Swap Transporteur,NaN,NaN,3min,Sofiane Aoues


In [375]:
df_clean.loc[
    df_clean["N° Ticket"] == "P260903270",
    "problematique"
] = "Swap Transporteur"

In [376]:
# Correction ciblée de la dernière ligne
mask_ticket = df_clean["N° Ticket"].eq("P260903270")

df_clean.loc[mask_ticket, "client"] = "ETAM"
df_clean.loc[mask_ticket, "problematique"] = "Swap Transporteur"
df_clean.loc[mask_ticket, "technicien"] = "Sofiane Aoues"

In [377]:
df_clean.loc[mask_ticket, "resolution_minutes"] = (
    df_clean.loc[mask_ticket, "Temps de résolution (h)"]
    .apply(parse_resolution_minutes)
)


In [378]:
df_clean["date_ouverture_normalisee"] = (
    df_clean["Date/Heure ouverture"]
    .apply(normalize_opening_datetime)
)

df_clean["date_heure_ouverture"] = (
    df_clean["date_ouverture_normalisee"]
    .apply(parse_opening_datetime)
)

df_clean["date_resolution_normalisee"] = (
    df_clean["Date/Heure résolution"]
    .apply(normalize_resolution_datetime)
)

df_clean["date_heure_resolution"] = (
    df_clean["date_resolution_normalisee"]
    .apply(parse_resolution_datetime)
)

In [379]:
df_clean.loc[
    df_clean["N° Ticket"].eq("P260903270")
].T

,679
3,
Date du ticket,14/09/2026
N° Ticket,P260903270
Client,Etam
Problématique,Swap Transporteur
Priorité,Basse
Date/Heure ouverture,14/09/2026 à 15:32
Date/Heure résolution,14/09/2026 à 15:35
Temps de résolution (h),3min
Technicien,Sofiane Aoues


In [380]:
# Vérification du ticket ajouté le 14/09/2026
df_clean.loc[
    df_clean["N° Ticket"].eq("P260903270")
].T

,679
3,
Date du ticket,14/09/2026
N° Ticket,P260903270
Client,Etam
Problématique,Swap Transporteur
Priorité,Basse
Date/Heure ouverture,14/09/2026 à 15:32
Date/Heure résolution,14/09/2026 à 15:35
Temps de résolution (h),3min
Technicien,Sofiane Aoues


In [381]:
colonnes_verification = [
    "N° Ticket",
    "client",
    "problematique",
    "priorite",
    "technicien",
    "resolution_minutes",
    "date_heure_ouverture",
    "date_heure_resolution",
    "heure_ouverture",
    "jour_ouverture_fr",
]

df_clean.loc[
    df_clean["N° Ticket"].eq("P260903270"),
    colonnes_verification
]

3,N° Ticket,client,problematique,priorite,technicien,resolution_minutes,date_heure_ouverture,date_heure_resolution,heure_ouverture,jour_ouverture_fr
679,P260903270,ETAM,Swap Transporteur,Basse,Sofiane Aoues,3.0,2026-09-14 15:32:00,2026-09-14 15:35:00,15,Lundi


In [382]:
# Contrôle des valeurs manquantes dans les variables analytiques
colonnes_analytiques = [
    "client",
    "problematique",
    "priorite",
    "technicien",
    "resolution_minutes",
    "date_heure_ouverture",
    "date_heure_resolution",
]

controle_final = pd.DataFrame({
    "Valeurs manquantes": df_clean[colonnes_analytiques].isna().sum(),
    "Valeurs renseignées": df_clean[colonnes_analytiques].notna().sum(),
})

controle_final["Taux de complétude (%)"] = (
    controle_final["Valeurs renseignées"] / len(df_clean) * 100
).round(2)

controle_final

,Valeurs manquantes,Valeurs renseignées,Taux de complétude (%)
3,,,
client,0,680,100.0
problematique,0,680,100.0
priorite,0,680,100.0
technicien,0,680,100.0
resolution_minutes,0,680,100.0
date_heure_ouverture,0,680,100.0
date_heure_resolution,0,680,100.0


In [383]:
print(f"Nombre de lignes analytiques : {len(df_clean)}")
print(f"Nombre de tickets renseignés : {df_clean['N° Ticket'].notna().sum()}")
print(f"Nombre de tickets uniques : {df_clean['N° Ticket'].nunique()}")

Nombre de lignes analytiques : 680
Nombre de tickets renseignés : 680
Nombre de tickets uniques : 673


In [384]:
# ==========================================
# 7. VALIDATION DE LA QUALITÉ DES DONNÉES
# ==========================================

nb_lignes = len(df_clean)

rapport_qualite = pd.DataFrame({
    "Indicateur": [
        "Nombre de lignes analytiques",
        "Nombre de tickets uniques",
        "Durées de résolution exploitables",
        "Dates d'ouverture exploitables",
        "Dates de résolution exploitables",
        "Lignes avec durée cohérente",
        "Lignes avec durée à vérifier",
        "Anomalies chronologiques",
    ],
    "Valeur": [
        nb_lignes,
        df_clean["N° Ticket"].nunique(),
        df_clean["resolution_minutes"].notna().sum(),
        df_clean["date_heure_ouverture"].notna().sum(),
        df_clean["date_heure_resolution"].notna().sum(),
        (df_clean["coherence_duree"] == "Cohérente").sum(),
        (df_clean["coherence_duree"] == "À vérifier").sum(),
        (
            df_clean["date_heure_resolution"]
            < df_clean["date_heure_ouverture"]
        ).sum(),
    ],
})

rapport_qualite

,Indicateur,Valeur
0,Nombre de lignes analytiques,680
1,Nombre de tickets uniques,673
2,Durées de résolution exploitables,680
3,Dates d'ouverture exploitables,680
4,Dates de résolution exploitables,680
5,Lignes avec durée cohérente,598
6,Lignes avec durée à vérifier,82
7,Anomalies chronologiques,16


In [385]:
rapport_qualite["Pourcentage"] = np.nan

indicateurs_avec_pourcentage = [
    "Durées de résolution exploitables",
    "Dates d'ouverture exploitables",
    "Dates de résolution exploitables",
    "Lignes avec durée cohérente",
]

rapport_qualite.loc[
    rapport_qualite["Indicateur"].isin(indicateurs_avec_pourcentage),
    "Pourcentage"
] = (
    rapport_qualite.loc[
        rapport_qualite["Indicateur"].isin(indicateurs_avec_pourcentage),
        "Valeur"
    ] / nb_lignes * 100
).round(2)

rapport_qualite

,Indicateur,Valeur,Pourcentage
0,Nombre de lignes analytiques,680,NaN
1,Nombre de tickets uniques,673,NaN
2,Durées de résolution exploitables,680,100.00
3,Dates d'ouverture exploitables,680,100.00
4,Dates de résolution exploitables,680,100.00
5,Lignes avec durée cohérente,598,87.94
6,Lignes avec durée à vérifier,82,NaN
7,Anomalies chronologiques,16,NaN


In [386]:
df_clean.head(5)

3,Date du ticket,N° Ticket,Client,Problématique,Priorité,Date/Heure ouverture,Date/Heure résolution,Temps de résolution (h),Technicien,date_ticket,resolution_minutes,technicien,priorite,annee,mois,mois_nom,semaine,jour_semaine,ouverture_normalisee,date_heure_ouverture,heure_ouverture,minute_ouverture,jour_ouverture,jour_ouverture_fr,resolution_normalisee,date_heure_resolution,resolution_datetime_valide,duree_calculee_minutes,ecart_duree_minutes,coherence_duree,client,problematique,date_ouverture_normalisee,date_resolution_normalisee
0,23/06/2026,P260604001,Etam,Fermeture provisoire : demande démontage,Faible,23/06/2026 à 9h40,23/06/2026 à 10h13,0h19min,Sofiane AOUES,2026-06-23,19.000000,Sofiane Aoues,Basse,2026,6,June,26,Tuesday,23/06/2026 9:40,2026-06-23 09:40:00,9,40,Tuesday,Mardi,23/06/2026 10:13,2026-06-23 10:13:00,True,33.0,14.000000,À vérifier,ETAM,Fermeture Provisoire,23/06/2026 9:40,23/06/2026 10:13
1,23/06/2026,P260604023,AMPLIFON,Changement poste,Elevé,23/06/2026 10:30:00,23/06/2026 à 10:44,41:14:00,LOUBNA,2026-06-23,41.233333,Loubna,Haute,2026,6,June,26,Tuesday,23/06/2026 10:30:00,2026-06-23 10:30:00,10,30,Tuesday,Mardi,23/06/2026 10:44,2026-06-23 10:44:00,True,14.0,-27.233333,À vérifier,AMPLIFON,Envoi Matériel,23/06/2026 10:30:00,23/06/2026 10:44
2,23/06/2026,P260604025,Amplifon,Envoi Matériel,Faible,23/06/2026 à 10h40,23/06/2026 à 10h48,0h5min,Sofiane AOUES,2026-06-23,5.000000,Sofiane Aoues,Basse,2026,6,June,26,Tuesday,23/06/2026 10:40,2026-06-23 10:40:00,10,40,Tuesday,Mardi,23/06/2026 10:48,2026-06-23 10:48:00,True,8.0,3.000000,À vérifier,AMPLIFON,Envoi Matériel,23/06/2026 10:40,23/06/2026 10:48
3,23/06/2026,P260604110,Etam,Ajout Matériel,Faible,23/06/2026 à 14h00,23/06/2026 à 14h12,0h10min,Sofiane AOUES,2026-06-23,10.000000,Sofiane Aoues,Basse,2026,6,June,26,Tuesday,23/06/2026 14:00,2026-06-23 14:00:00,14,0,Tuesday,Mardi,23/06/2026 14:12,2026-06-23 14:12:00,True,12.0,2.000000,Cohérente,ETAM,Ajout Matériel,23/06/2026 14:00,23/06/2026 14:12
4,23/06/2026,P260604148,Pos Service,Maintenance matériel,Basse,23/06/2026 à 16h25,23/06/2026 à 16h40,0h13min,Sofiane Aoues,2026-06-23,13.000000,Sofiane Aoues,Basse,2026,6,June,26,Tuesday,23/06/2026 16:25,2026-06-23 16:25:00,16,25,Tuesday,Mardi,23/06/2026 16:40,2026-06-23 16:40:00,True,15.0,2.000000,Cohérente,POS SERVICE,Maintenance,23/06/2026 16:25,23/06/2026 16:40


Étape 8 — Statistiques descriptives

Nous allons commencer par créer un bloc statistique propre et réutilisable.

8.1 Calculer les indicateurs

In [387]:
# ==========================================
# 8. STATISTIQUES DESCRIPTIVES
# ==========================================

from scipy.stats import skew

durees = df_clean["resolution_minutes"].dropna()

statistiques_resolution = {
    "Total des lignes analytiques": len(durees),
    "Temps moyen (min)": durees.mean(),
    "Temps médian (min)": durees.median(),
    "Écart-type (min)": durees.std(),
    "Q1 (min)": durees.quantile(0.25),
    "Q3 (min)": durees.quantile(0.75),
    "IQR (min)": durees.quantile(0.75) - durees.quantile(0.25),
    "P95 (min)": durees.quantile(0.95),
    "Minimum (min)": durees.min(),
    "Maximum (min)": durees.max(),
    "Asymétrie — Skewness": skew(durees),
}

statistiques_resolution_df = (
    pd.DataFrame.from_dict(
        statistiques_resolution,
        orient="index",
        columns=["Valeur"]
    )
)

statistiques_resolution_df

,Valeur
Total des lignes analytiques,680.000000
Temps moyen (min),4.873211
Temps médian (min),4.000000
Écart-type (min),4.116082
Q1 (min),3.000000
Q3 (min),5.000000
IQR (min),2.000000
P95 (min),11.000000
Minimum (min),1.000000
Maximum (min),41.233333


In [388]:
def formater_minutes(valeur, decimales=2):
    """
    Formate une durée exprimée en minutes.
    """
    if pd.isna(valeur):
        return "N/A"

    return f"{valeur:.{decimales}f} min"

In [389]:
formater_minutes(statistiques_resolution["Temps moyen (min)"])
formater_minutes(statistiques_resolution["Temps médian (min)"])
formater_minutes(statistiques_resolution["P95 (min)"])

'11.00 min'

## 05  Statistiques Descriptives

In [390]:
# ==========================================
# 8.3 CARTES KPI
# ==========================================

from IPython.display import HTML, display
from html import escape


def creer_carte_kpi(titre, valeur, sous_titre, classe_css):
    """
    Génère le HTML d'une carte KPI.
    """
    return f"""
    <div class="kpi-card {classe_css}">
        <div class="kpi-title">{escape(titre)}</div>
        <div class="kpi-value">{escape(valeur)}</div>
        <div class="kpi-subtitle">{escape(sous_titre)}</div>
    </div>
    """


total_lignes = len(durees)
temps_moyen = formater_minutes(durees.mean())
temps_median = formater_minutes(durees.median())
p95 = formater_minutes(durees.quantile(0.95))

kpi_html = f"""
<style>
.kpi-container {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));
    gap: 18px;
    margin: 20px 0 30px 0;
    font-family: Arial, sans-serif;
}}

.kpi-card {{
    padding: 22px;
    border-radius: 14px;
    color: white;
    min-height: 125px;
    box-shadow: 0 5px 14px rgba(0, 0, 0, 0.12);
    display: flex;
    flex-direction: column;
    justify-content: space-between;
}}

.kpi-card-primary {{
    background: linear-gradient(135deg, #4338ca, #6366f1);
}}

.kpi-card-secondary {{
    background: linear-gradient(135deg, #0369a1, #0ea5e9);
}}

.kpi-card-success {{
    background: linear-gradient(135deg, #047857, #10b981);
}}

.kpi-card-warning {{
    background: linear-gradient(135deg, #b45309, #f59e0b);
}}

.kpi-title {{
    font-size: 13px;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.8px;
    opacity: 0.92;
}}

.kpi-value {{
    font-size: 30px;
    font-weight: 700;
    margin: 10px 0;
    line-height: 1.2;
}}

.kpi-subtitle {{
    font-size: 12px;
    opacity: 0.85;
    line-height: 1.4;
}}
</style>

<div class="kpi-container">
    {creer_carte_kpi(
        "Lignes analytiques",
        f"{total_lignes}",
        "Unité d'analyse retenue",
        "kpi-card-primary"
    )}

    {creer_carte_kpi(
        "Temps moyen",
        temps_moyen,
        "Moyenne des durées déclarées",
        "kpi-card-secondary"
    )}

    {creer_carte_kpi(
        "Temps médian",
        temps_median,
        "Indicateur robuste",
        "kpi-card-success"
    )}

    {creer_carte_kpi(
        "P95",
        p95,
        "95e percentile des durées",
        "kpi-card-warning"
    )}
</div>
"""

display(HTML(kpi_html))

In [391]:
# ==========================================
# 8.4 INTERPRÉTATION DES STATISTIQUES
# ==========================================

ratio_moyenne_mediane = durees.mean() / durees.median()
skewness_valeur = skew(durees)

interpretation = f"""
### Lecture des temps de résolution

- Le périmètre comprend **{len(durees)} lignes analytiques**.
- Le temps médian de résolution est de **{durees.median():.2f} minutes**.
- Le temps moyen est de **{durees.mean():.2f} minutes**.
- Le 95e percentile est de **{durees.quantile(0.95):.2f} minutes**.
- L'écart entre la moyenne et la médiane indique que certaines durées
  plus élevées influencent la moyenne.
- L'asymétrie observée est de **{skewness_valeur:.2f}**, ce qui traduit
  une distribution étirée vers les valeurs élevées.

Les durées sont donc majoritairement concentrées sur des valeurs faibles,
avec un nombre limité d'observations plus longues. Ces observations doivent
être examinées séparément avant toute interprétation opérationnelle.
"""

display(HTML(interpretation))

Distribution des temps de résolution

In [392]:
# ==========================================
# 9.1 PRÉPARATION DE LA DISTRIBUTION
# ==========================================

durees = (
    df_clean["resolution_minutes"]
    .dropna()
    .astype(float)
)

temps_moyen = durees.mean()
temps_median = durees.median()
temps_p95 = durees.quantile(0.95)

print(f"Nombre d'observations : {len(durees)}")
print(f"Temps moyen : {temps_moyen:.2f} min")
print(f"Temps médian : {temps_median:.2f} min")
print(f"P95 : {temps_p95:.2f} min")

Nombre d'observations : 680
Temps moyen : 4.87 min
Temps médian : 4.00 min
P95 : 11.00 min


In [393]:
import plotly.graph_objects as go

fig_distribution = go.Figure()

# 1. Histogramme avec un design plus moderne (bleu pro et bordures nettes)
fig_distribution.add_trace(
    go.Histogram(
        x=durees,
        xbins=dict(
            start=0,
            end=45,
            size=5
        ),
        name="Tickets",
        marker=dict(
            color="#3b82f6",  # Bleu vif moderne
            line=dict(color="#1d4ed8", width=1) # Bordure plus sombre pour détacher les barres
        ),
        opacity=0.85,
        hovertemplate=(
            "<b>Intervalle :</b> %{x} min<br>"
            "<b>Nombre de tickets :</b> %{y}<br>"
            "<extra></extra>"
        )
    )
)

# 2. Ligne de la Médiane (Vert émeraude - Indicateur central robuste)
fig_distribution.add_vline(
    x=temps_median,
    line_dash="dot",
    line_width=2.5,
    line_color="#10b981",
    annotation_text=f"Médiane : {temps_median:.1f} min",
    annotation_position="top left",
    annotation=dict(font_size=11, font_color="#065f46", y=0.92)
)

# 3. Ligne de la Moyenne (Orange ambré)
fig_distribution.add_vline(
    x=temps_moyen,
    line_dash="dash",
    line_width=2,
    line_color="#f59e0b",
    annotation_text=f"Moyenne : {temps_moyen:.1f} min",
    annotation_position="top right",
    annotation=dict(font_size=11, font_color="#92400e", y=0.82)
)

# 4. Ligne du P95 (Rouge alerte - Seuil critique)
fig_distribution.add_vline(
    x=temps_p95,
    line_dash="dashdot",
    line_width=2,
    line_color="#ef4444",
    annotation_text=f"P95 : {temps_p95:.1f} min",
    annotation_position="top right",
    annotation=dict(font_size=11, font_color="#991b1b", y=0.72)
)

# 5. Mise en page globale harmonisée
fig_distribution.update_layout(
    title=dict(
        text="<b>Distribution des temps de résolution</b><br><span style='font-size:12px; color:#64748b;'>Analyse de la dispersion, de la tendance centrale et du seuil critique (P95)</span>",
        font=dict(size=16, color="#1e293b", family="Arial"),
        x=0.0,
        y=0.92,
    ),
    xaxis=dict(
        title="Temps de résolution (minutes)",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="Nombre de tickets",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    template="plotly_white",
    height=480,
    bargap=0.05, # Resserre légèrement les barres pour un aspect plus compact et professionnel
    hovermode="x unified",
    showlegend=False, # Masqué car les annotations textuelles identifient déjà clairement chaque ligne
    margin=dict(t=80, b=40, l=60, r=40),
)

fig_distribution.show()

### Interprétation

La distribution des temps de résolution est concentrée sur les durées courtes
et présente une asymétrie positive. La moyenne est influencée par un nombre
limité de durées plus élevées, ce qui justifie l'utilisation conjointe de la
moyenne, de la médiane et du 95e percentile.

Le P95 constitue un repère descriptif permettant d'observer le comportement
des 5 % de durées les plus élevées. Il ne constitue pas un seuil SLA, aucun
objectif de service n'ayant encore été défini.

In [394]:
# ==========================================
# 9.4 EXPORT HTML INTERACTIF
# ==========================================

from pathlib import Path

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig_distribution.write_html(
    FIGURES_DIR / "distribution_temps_resolution.html",
    include_plotlyjs=True,
    full_html=True,
)

print("Graphique HTML exporté avec succès.")

Graphique HTML exporté avec succès.


In [395]:
# ============================================================
# 9.1 — Distribution des temps de résolution
# ============================================================

durees = df_clean["resolution_minutes"].dropna().astype(float)

temps_moyen = durees.mean()
temps_median = durees.median()
temps_p95 = durees.quantile(0.95)

# Classes de durée en minutes
bornes = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]

labels = [
    "0–5 min",
    "5–10 min",
    "10–15 min",
    "15–20 min",
    "20–25 min",
    "25–30 min",
    "30–35 min",
    "35–40 min",
    "40–45 min",
]

df_distribution = pd.DataFrame({
    "duree_minutes": durees
})

df_distribution["classe_duree"] = pd.cut(
    df_distribution["duree_minutes"],
    bins=bornes,
    labels=labels,
    right=False,
    include_lowest=True
)

distribution = (
    df_distribution["classe_duree"]
    .value_counts(sort=False)
    .rename_axis("Classe de durée")
    .reset_index(name="Nombre de lignes")
)

distribution["Pourcentage"] = (
    distribution["Nombre de lignes"] / len(durees) * 100
).round(1)

distribution

,Classe de durée,Nombre de lignes,Pourcentage
0,0–5 min,452,66.5
1,5–10 min,168,24.7
2,10–15 min,41,6.0
3,15–20 min,10,1.5
4,20–25 min,2,0.3
5,25–30 min,3,0.4
6,30–35 min,1,0.1
7,35–40 min,1,0.1
8,40–45 min,2,0.3


In [396]:
import plotly.express as px

# 1. Création du graphique en barres avec une couleur professionnelle
fig_distribution = px.bar(
    distribution,
    x="Classe de durée",
    y="Nombre de lignes",
    text="Nombre de lignes",
    title=(
        "<b>Distribution des temps de résolution</b>"
        f"<br><span style='font-size:12px; color:#64748b;'>Moyenne : {temps_moyen:.2f} min — "
        f"Médiane : {temps_median:.2f} min — "
        f"P95 : {temps_p95:.2f} min</span>"
    ),
    labels={
        "Classe de durée": "Temps de résolution (intervalles)",
        "Nombre de lignes": "Nombre de lignes analytiques"
    },
    hover_data={
        "Pourcentage": True
    }
)

# 2. Personnalisation avancée de l'apparence des barres et des infobulles
fig_distribution.update_traces(
    textposition="outside",
    marker=dict(
        color="#3b82f6",  # Bleu pro
        line=dict(color="#1d4ed8", width=1) # Contour plus sombre pour du relief
    ),
    hovertemplate=(
        "<b>Intervalle :</b> %{x}<br>"
        "<b>Volume :</b> %{y} lignes<br>"
        "<b>Part :</b> %{customdata[0]:.1f}%"
        "<extra></extra>"
    )
)

# 3. Mise en page aux standards professionnels
fig_distribution.update_layout(
    template="plotly_white",
    height=500,
    width=1100,
    title={
        "x": 0.0,
        "xanchor": "left" # Titre aligné à gauche (plus moderne et lisible qu'un centrage)
    },
    xaxis=dict(
        title="Temps de résolution",
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="Nombre de lignes analytiques",
        showgrid=True,
        gridcolor="#e2e8f0", # Grille subtile pour la lecture horizontale
        linecolor="#cbd5e1",
        rangemode="tozero",
        autorange=True # Laisse de l'espace en haut pour les textes "outside"
    ),
    margin=dict(t=90, b=60, l=70, r=40),
    showlegend=False
)

fig_distribution.show()

In [397]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig_distribution.write_html(
    FIGURES_DIR / "distribution_temps_resolution.html",
    include_plotlyjs=True,
    full_html=True
)

try:
    fig_distribution.write_image(
        FIGURES_DIR / "distribution_temps_resolution.png",
        width=1400,
        height=800,
        scale=2
    )
    print("Export PNG réussi.")
except Exception as e:
    print("Export PNG impossible :", e)
    print("Vérifie que le package kaleido est installé.")

Export PNG réussi.


### Interprétation

La distribution des temps de résolution montre une concentration importante des lignes
analytiques dans les premières classes de durée.

La médiane est inférieure à la moyenne, ce qui indique une distribution asymétrique
avec la présence de quelques durées plus élevées.

Le 95e percentile permet d'identifier le niveau de durée en dessous duquel se situent
environ 95 % des lignes analytiques.

Les temps de résolution utilisés dans cette analyse proviennent directement du champ
`Temps de résolution (h)`, considéré comme la référence opérationnelle. Les dates et
heures d'ouverture et de résolution n'ont pas été utilisées pour recalculer les durées.

In [398]:
# ============================================================
# 9.2 — Évolution du volume de lignes analytiques par jour
# ============================================================

volume_quotidien = (
    df_clean
    .groupby("date_ticket")
    .size()
    .rename("nombre_lignes")
    .reset_index()
    .sort_values("date_ticket")
)

volume_quotidien.head()

,date_ticket,nombre_lignes
0,2026-06-23,5
1,2026-06-24,14
2,2026-06-25,13
3,2026-06-26,13
4,2026-06-29,13


In [399]:
print("Première date :", volume_quotidien["date_ticket"].min().date())
print("Dernière date :", volume_quotidien["date_ticket"].max().date())
print("Nombre de jours actifs :", len(volume_quotidien))
print("Moyenne quotidienne :", volume_quotidien["nombre_lignes"].mean().round(2))

Première date : 2026-06-23
Dernière date : 2026-09-14
Nombre de jours actifs : 53
Moyenne quotidienne : 12.83


In [400]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# 1. Calcul d'une moyenne mobile sur 7 jours pour lisser l'effet hebdomadaire
volume_quotidien["moyenne_mobile"] = volume_quotidien["nombre_lignes"].rolling(window=7, center=True).mean()

# 2. Création de la figure de base avec Plotly Express
fig_volume_quotidien = px.line(
    volume_quotidien,
    x="date_ticket",
    y="nombre_lignes",
    markers=True,
    labels={
        "date_ticket": "Date",
        "nombre_lignes": "Nombre de tickets",
    },
)

# 3. Personnalisation de la courbe principale (couleur bleu pro, style épuré)
fig_volume_quotidien.update_traces(
    selector=dict(mode="lines+markers"),
    line=dict(color="#1e3a8a", width=2),
    marker=dict(size=4, color="#1e3a8a"),
    hovertemplate=(
        "<b>Date :</b> %{x|%d/%m/%Y}<br>"
        "<b>Volume journalier :</b> %{y} tickets<br>"
        "<extra></extra>"
    ),
)

# 4. Ajout de la Moyenne Mobile (courbe lisse en arrière-plan)
fig_volume_quotidien.add_trace(
    go.Scatter(
        x=volume_quotidien["date_ticket"],
        y=volume_quotidien["moyenne_mobile"],
        name="Tendance (Moy. Mobile 7j)",
        mode="lines",
        line=dict(color="#f87171", width=3, shape="spline"), # Rouge corail élégant et lisse
        hovertemplate=(
            "<b>Tendance (7j) :</b> %{y:.1f} tickets<br>"
            "<extra></extra>"
        ),
    )
)

# 5. Mise en page aux standards professionnels
fig_volume_quotidien.update_layout(
    title=dict(
        text="<b>Évolution quotidienne du volume de tickets</b><br><span style='font-size:12px; color:#64748b;'>Suivi journalier et lissage de la tendance par moyenne mobile sur 7 jours</span>",
        font=dict(size=16, color="#1e293b", family="Arial"),
        x=0.0,
        y=0.92,
    ),
    template="plotly_white",
    height=450,
    xaxis=dict(
        tickformat="%d/%m/%Y",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    margin=dict(t=80, b=40, l=60, r=30),
    showlegend=True,
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=1.02, 
        xanchor="right", 
        x=1,
        bgcolor="rgba(0,0,0,0)"
    ),
    hovermode="x unified" # Permet de voir d'un coup d'œil le volume réel et la moyenne mobile au survol
)

fig_volume_quotidien.show()

In [401]:
fig_volume_quotidien.write_html(
    FIGURES_DIR / "volume_quotidien.html",
    include_plotlyjs=True,
    full_html=True
)

try:
    fig_volume_quotidien.write_image(
        FIGURES_DIR / "volume_quotidien.png",
        width=1400,
        height=800,
        scale=2
    )
    print("Export PNG réussi.")
except Exception as e:
    print("Export PNG impossible :", e)

Export PNG réussi.


### Interprétation

L’évolution quotidienne permet de suivre les variations du volume de lignes
analytiques enregistrées sur la période étudiée.

Les fluctuations observées peuvent refléter des variations de charge opérationnelle,
mais également des différences de saisie ou de couverture du suivi.

Ce graphique doit donc être interprété comme un indicateur du volume enregistré,
et non comme une mesure exhaustive de la charge réelle du Help Desk.

In [402]:
# ============================================================
# 9.3 — Volume de lignes analytiques par client
# ============================================================

volume_clients = (
    df_clean
    .groupby("client")
    .size()
    .rename("nombre_lignes")
    .reset_index()
    .sort_values("nombre_lignes", ascending=True)
)

volume_clients["pourcentage"] = (
    volume_clients["nombre_lignes"]
    / volume_clients["nombre_lignes"].sum()
    * 100
).round(1)

volume_clients

,client,nombre_lignes,pourcentage
6,CCMO,1,0.1
11,OXHOO,1,0.1
10,L'ADDITION,1,0.1
14,PROMETHEAN,2,0.3
12,PIXEL INSPIRATION,4,0.6
4,BARRON MCCANN,5,0.7
9,INNOVORDER,6,0.9
3,AXE E-SANTE,11,1.6
5,BUT,48,7.1
13,POS SERVICE,52,7.6


In [466]:
import plotly.express as px

# 1. Optionnel mais recommandé : trier par ordre décroissant pour un affichage propre en haut du graphique
# (Si ton DataFrame 'volume_clients' n'est pas déjà trié)
volume_clients = volume_clients.sort_values("nombre_lignes", ascending=True) 

# 2. Création du graphique en barres horizontales
fig_volume_clients = px.bar(
    volume_clients,
    x="nombre_lignes",
    y="client",
    orientation="h",
    text="nombre_lignes",
    title=(
        "<b>Volume de tickets par client</b>"
        "<br><span style='font-size:12px; color:#64748b;'>Répartition du volume global et parts relatives par client</span>"
    ),
    labels={
        "client": "Client",
        "nombre_lignes": "Nombre de tickets"
    },
    hover_data={
        "pourcentage": ":.1f"
    }
)

# 3. Personnalisation avancée des barres et des infobulles
fig_volume_clients.update_traces(
    textposition="outside",
    cliponaxis=False,
    marker=dict(
        color="#3b82f6",  # Bleu pro
        line=dict(color="#1d4ed8", width=1) # Contour net pour du relief
    ),
    hovertemplate=(
        "<b>Client :</b> %{y}<br>"
        "<b>Nombre de lignes :</b> %{x}<br>"
        "<b>Part du volume :</b> %{customdata[0]:.1f}%"
        "<extra></extra>"
    )
)

# 4. Mise en page aux standards professionnels
fig_volume_clients.update_layout(
    template="plotly_white",
    height=max(500, len(volume_clients) * 35),
    width=1100,
    title={
        "x": 0.0,
        "xanchor": "left"  # Titre aligné à gauche, plus élégant
    },
    xaxis=dict(
        title="Nombre de tickets",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0", # Grille verticale pour faciliter la lecture des valeurs
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=volume_clients["client"].tolist(),
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    margin=dict(t=90, b=60, l=160, r=100), # Marge gauche confortable pour les noms de clients
    showlegend=False
)

fig_volume_clients.show()

In [404]:
# Export HTML
fig_volume_clients.write_html(
    FIGURES_DIR / "volume_par_client.html",
    include_plotlyjs=True,
    full_html=True
)

# Export PNG
try:
    fig_volume_clients.write_image(
        FIGURES_DIR / "volume_par_client.png",
        width=1400,
        height=max(800, len(volume_clients) * 45),
        scale=2
    )
    print("Export PNG réussi.")
except Exception as e:
    print("Export PNG impossible :", e)

Export PNG réussi.


### Interprétation

La répartition du volume de lignes analytiques met en évidence les clients
représentant la plus grande part de l'activité enregistrée du Help Desk.

Cette répartition permet d'identifier les principaux contributeurs au volume
de demandes et peut servir de base à une analyse plus approfondie des temps
de résolution par client.

Le volume de lignes ne constitue toutefois pas, à lui seul, un indicateur
de complexité ou de charge réelle : deux lignes peuvent correspondre à des
interventions de nature très différente.

In [405]:
# ============================================================
# 9.4 — Temps de résolution par client
# ============================================================

stats_clients = (
    df_clean
    .groupby("client")
    .agg(
        nombre_lignes=("resolution_minutes", "count"),
        temps_moyen=("resolution_minutes", "mean"),
        temps_median=("resolution_minutes", "median"),
        temps_p95=("resolution_minutes", lambda x: x.quantile(0.95))
    )
    .reset_index()
)

stats_clients["temps_moyen"] = stats_clients["temps_moyen"].round(2)
stats_clients["temps_median"] = stats_clients["temps_median"].round(2)
stats_clients["temps_p95"] = stats_clients["temps_p95"].round(2)

stats_clients = stats_clients.sort_values(
    "temps_median",
    ascending=True
)

stats_clients

,client,nombre_lignes,temps_moyen,temps_median,temps_p95
1,AEM SOFT,66,5.06,3.0,12.00
2,AMPLIFON,59,4.04,3.0,9.10
7,CENTRIC ACTION,112,3.49,3.0,7.00
5,BUT,48,4.35,3.0,8.00
15,SHOPPERTRAK,73,4.92,3.0,12.40
11,OXHOO,1,3.00,3.0,3.00
6,CCMO,1,4.00,4.0,4.00
10,L'ADDITION,1,4.00,4.0,4.00
13,POS SERVICE,52,5.29,4.0,10.00
12,PIXEL INSPIRATION,4,4.25,4.0,5.70


In [406]:
import plotly.express as px

# 1. Transformation en format long en ne gardant que la Moyenne et la Médiane
stats_clients_long = stats_clients.melt(
    id_vars=["client", "nombre_lignes"],
    value_vars=[
        "temps_moyen",
        "temps_median"
    ],
    var_name="indicateur",
    value_name="duree_minutes"
)

noms_indicateurs = {
    "temps_moyen": "Moyenne",
    "temps_median": "Médiane"
}

stats_clients_long["indicateur"] = (
    stats_clients_long["indicateur"]
    .map(noms_indicateurs)
)

# 2. Création du graphique groupé horizontal
fig_temps_clients = px.bar(
    stats_clients_long,
    x="duree_minutes",
    y="client",
    color="indicateur",
    barmode="group",
    orientation="h",
    text="duree_minutes",  # Affiche la valeur numérique
    title=(
        "<b>Temps de résolution par client (Moyenne vs Médiane)</b>"
        "<br><span style='font-size:12px; color:#64748b;'>Comparaison des tendances centrales par client</span>"
    ),
    labels={
        "client": "Client",
        "duree_minutes": "Durée de résolution (minutes)",
        "indicateur": "Indicateur"
    },
    color_discrete_map={
        "Moyenne": "#3b82f6",  # Bleu pro
        "Médiane": "#f87171"   # Rouge corail élégant (harmonisé avec le reste de ton rapport)
    },
    hover_data={
        "nombre_lignes": True,
        "duree_minutes": ":.2f"
    }
)

# 3. Personnalisation des textes et des barres
fig_temps_clients.update_traces(
    texttemplate='%{text:.1f} min',  # Formate le texte (1 chiffre après la virgule + "min")
    textposition="outside",          # Place le texte juste à côté de la barre
    cliponaxis=False,                # Empêche le texte d'être coupé s'il dépasse
    hovertemplate=(
        "<b>Client :</b> %{y}<br>"
        "<b>%{fullData.name} :</b> %{x:.2f} min<br>"
        "<b>Volume lignes :</b> %{customdata[0]}"
        "<extra></extra>"
    )
)

# 4. Mise en page aux standards professionnels
fig_temps_clients.update_layout(
    template="plotly_white",
    height=max(500, len(stats_clients) * 45),  # Ajuste dynamiquement la hauteur selon le nombre de clients
    width=1200,
    title={
        "x": 0.0,
        "xanchor": "left"
    },
    xaxis=dict(
        title="Durée de résolution (minutes)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=stats_clients["client"].tolist(),
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    margin=dict(t=90, b=70, l=160, r=80),  # Marge droite élargie pour laisser la place aux textes "outside"
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    ),
    legend_title_text=""
)

fig_temps_clients.show()

In [407]:
# Export HTML
fig_temps_clients.write_html(
    FIGURES_DIR / "temps_resolution_par_client.html",
    include_plotlyjs=True,
    full_html=True
)

# Export PNG
try:
    fig_temps_clients.write_image(
        FIGURES_DIR / "temps_resolution_par_client.png",
        width=1500,
        height=max(850, len(stats_clients) * 50),
        scale=2
    )
    print("Export PNG réussi.")
except Exception as e:
    print("Export PNG impossible :", e)

Export PNG réussi.


### Interprétation

Les temps de résolution sont comparés entre les clients à partir de trois indicateurs :
la moyenne, la médiane et le 95e percentile.

La médiane permet d'appréhender le temps de résolution typique, tandis que la moyenne
est davantage sensible aux durées élevées. Le P95 fournit une indication sur le niveau
de durée atteint par les observations les plus longues.

Les résultats doivent être interprétés conjointement avec le nombre de lignes
analytiques par client. Les clients ayant un faible volume d'activité peuvent présenter
des indicateurs plus instables.

Ces indicateurs décrivent les durées déclarées et ne constituent pas une mesure de
performance comparative entre clients. Aucun seuil SLA n'étant encore défini, le P95
ne doit pas être interprété comme un taux de conformité SLA.

In [408]:
# ============================================================
# 9.5 — Volume de lignes analytiques par problématique
# ============================================================

volume_problematiques = (
    df_clean
    .groupby("problematique")
    .size()
    .rename("nombre_lignes")
    .reset_index()
    .sort_values("nombre_lignes", ascending=True)
)

volume_problematiques["pourcentage"] = (
    volume_problematiques["nombre_lignes"]
    / volume_problematiques["nombre_lignes"].sum()
    * 100
).round(1)

volume_problematiques

,problematique,nombre_lignes,pourcentage
4,Fermeture Provisoire,4,0.6
9,Intervention Date Imposée,4,0.6
14,Ramassage Matériel,4,0.6
1,Dismantling Old Store,6,0.9
3,Fermeture Définitive,8,1.2
0,Ajout Matériel,9,1.3
8,Installation New Store,13,1.9
15,Remplacement Matériel,17,2.5
5,Floorwalker,22,3.2
6,Handover,22,3.2


In [409]:
import plotly.express as px

# 1. Optionnel mais recommandé : trier par ordre croissant pour que la plus grande barre soit en haut
volume_problematiques = volume_problematiques.sort_values("nombre_lignes", ascending=True)

# 2. Création du graphique en barres horizontales
fig_volume_problematiques = px.bar(
    volume_problematiques,
    x="nombre_lignes",
    y="problematique",
    orientation="h",
    text="nombre_lignes",
    title=(
        "<b>Volume de lignes analytiques par problématique</b>"
        "<br><span style='font-size:12px; color:#64748b;'>Répartition des volumes et parts relatives par type de problématique</span>"
    ),
    labels={
        "problematique": "Problématique",
        "nombre_lignes": "Nombre de lignes analytiques"
    },
    hover_data={
        "pourcentage": ":.1f"
    }
)

# 3. Personnalisation avancée des barres et des infobulles
fig_volume_problematiques.update_traces(
    textposition="outside",
    cliponaxis=False,
    marker=dict(
        color="#3b82f6",  # Bleu pro
        line=dict(color="#1d4ed8", width=1) # Contour net pour du relief
    ),
    hovertemplate=(
        "<b>Problématique :</b> %{y}<br>"
        "<b>Nombre de lignes :</b> %{x}<br>"
        "<b>Part du volume :</b> %{customdata[0]:.1f}%"
        "<extra></extra>"
    )
)

# 4. Mise en page aux standards professionnels
fig_volume_problematiques.update_layout(
    template="plotly_white",
    height=max(600, len(volume_problematiques) * 35),
    width=1200,
    title={
        "x": 0.0,
        "xanchor": "left"  # Titre aligné à gauche pour un rendu plus moderne
    },
    xaxis=dict(
        title="Nombre de lignes analytiques",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0", # Grille verticale pour faciliter la lecture
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=volume_problematiques["problematique"].tolist(),
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    margin=dict(t=90, b=70, l=230, r=100), # Marge gauche large préservée pour les libellés
    showlegend=False
)

fig_volume_problematiques.show()

In [410]:
# Export HTML
fig_volume_problematiques.write_html(
    FIGURES_DIR / "volume_par_problematique.html",
    include_plotlyjs=True,
    full_html=True
)

# Export PNG
try:
    fig_volume_problematiques.write_image(
        FIGURES_DIR / "volume_par_problematique.png",
        width=1600,
        height=max(900, len(volume_problematiques) * 45),
        scale=2
    )
    print("Export PNG réussi.")
except Exception as e:
    print("Export PNG impossible :", e)

Export PNG réussi.


### Interprétation

La répartition des lignes analytiques par problématique permet d'identifier les
principales catégories d'intervention enregistrées dans le suivi Help Desk.

Les catégories les plus représentées concentrent une part importante du volume
d'activité et peuvent constituer des axes prioritaires pour l'analyse opérationnelle.

Les catégories à faible volume doivent toutefois être interprétées avec prudence :
leur faible fréquence ne signifie pas nécessairement qu'elles sont peu importantes.

Les catégories utilisées dans cette analyse proviennent du champ normalisé
`problematique`.

In [411]:
# ============================================================
# 9.6 — Diagramme de Pareto des problématiques
# ============================================================

pareto_problematiques = (
    df_clean
    .groupby("problematique")
    .size()
    .rename("nombre_lignes")
    .reset_index()
    .sort_values("nombre_lignes", ascending=False)
    .reset_index(drop=True)
)

# Part de chaque catégorie dans le volume total
pareto_problematiques["pourcentage"] = (
    pareto_problematiques["nombre_lignes"]
    / pareto_problematiques["nombre_lignes"].sum()
    * 100
)

# Pourcentage cumulé
pareto_problematiques["pourcentage_cumule"] = (
    pareto_problematiques["pourcentage"].cumsum()
)

pareto_problematiques["rang"] = (
    pareto_problematiques.index + 1
)

pareto_problematiques

,problematique,nombre_lignes,pourcentage,pourcentage_cumule,rang
0,Maintenance,197,28.970588,28.970588,1
1,Swap Transporteur,95,13.970588,42.941176,2
2,Installation,89,13.088235,56.029412,3
3,Envoi Matériel,73,10.735294,66.764706,4
4,Intervention J+1,57,8.382353,75.147059,5
5,Intervention sur Site,32,4.705882,79.852941,6
6,Opening Assistance,28,4.117647,83.970588,7
7,Handover,22,3.235294,87.205882,8
8,Floorwalker,22,3.235294,90.441176,9
9,Remplacement Matériel,17,2.500000,92.941176,10


In [412]:
pareto_problematiques.head(10)

,problematique,nombre_lignes,pourcentage,pourcentage_cumule,rang
0,Maintenance,197,28.970588,28.970588,1
1,Swap Transporteur,95,13.970588,42.941176,2
2,Installation,89,13.088235,56.029412,3
3,Envoi Matériel,73,10.735294,66.764706,4
4,Intervention J+1,57,8.382353,75.147059,5
5,Intervention sur Site,32,4.705882,79.852941,6
6,Opening Assistance,28,4.117647,83.970588,7
7,Handover,22,3.235294,87.205882,8
8,Floorwalker,22,3.235294,90.441176,9
9,Remplacement Matériel,17,2.500000,92.941176,10


In [413]:
ligne_80 = pareto_problematiques[
    pareto_problematiques["pourcentage_cumule"] >= 80
].iloc[0]

print(
    f"Le seuil de 80 % est atteint après "
    f"{int(ligne_80['rang'])} problématiques."
)

print(
    f"Couverture obtenue : "
    f"{ligne_80['pourcentage_cumule']:.1f} %"
)

Le seuil de 80 % est atteint après 7 problématiques.
Couverture obtenue : 84.0 %


In [414]:
import plotly.graph_objects as go

# ============================================================
# Création du diagramme de Pareto optimisé
# ============================================================

fig_pareto = go.Figure()

# 1. Barres : volume par problématique
fig_pareto.add_trace(
    go.Bar(
        x=pareto_problematiques["problematique"],
        y=pareto_problematiques["nombre_lignes"],
        name="Volume (lignes)",
        text=pareto_problematiques["nombre_lignes"],
        textposition="outside",
        marker=dict(
            color="#3b82f6",  # Bleu pro
            line=dict(color="#1d4ed8", width=1) # Contour net pour du relief
        ),
        hovertemplate=(
            "<b>Problématique :</b> %{x}<br>"
            "<b>Nombre de lignes :</b> %{y}<br>"
            "<extra></extra>"
        )
    )
)

# 2. Courbe : pourcentage cumulé (sur l'axe secondaire y2)
fig_pareto.add_trace(
    go.Scatter(
        x=pareto_problematiques["problematique"],
        y=pareto_problematiques["pourcentage_cumule"],
        name="Pourcentage cumulé",
        mode="lines+markers",
        yaxis="y2",
        line=dict(color="#f87171", width=3), # Rouge corail élégant
        marker=dict(size=6, color="#f87171"),
        hovertemplate=(
            "<b>Problématique :</b> %{x}<br>"
            "<b>Cumul :</b> %{y:.1f}%<br>"
            "<extra></extra>"
        )
    )
)

# 3. Ligne de référence à 80 % (Seuil de Pareto)
fig_pareto.add_hline(
    y=80,
    yref="y2",
    line_dash="dash",
    line_width=1.5,
    line_color="#64748b", # Gris discret
    annotation_text="Seuil critique (80%)",
    annotation_position="top right",
    annotation=dict(font_size=11, font_color="#64748b")
)

# 4. Mise en page globale aux standards professionnels
fig_pareto.update_layout(
    template="plotly_white",
    height=650,
    width=1200,
    title=dict(
        text=(
            "<b>Diagramme de Pareto des problématiques (Règle des 80/20)</b>"
            "<br><span style='font-size:12px; color:#64748b;'>Identification des causes principales représentant 80% du volume total</span>"
        ),
        font=dict(size=16, color="#1e293b", family="Arial"),
        x=0.0,
        y=0.92
    ),
    xaxis=dict(
        title="Problématique",
        tickangle=-45, # Légèrement redressé à -45° pour un meilleur confort de lecture
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="Nombre de lignes analytiques",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis2=dict(
        title="Pourcentage cumulé (%)",
        overlaying="y",
        side="right",
        range=[0, 105],
        ticksuffix="%",
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    margin=dict(t=100, b=160, l=80, r=80),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.04,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    ),
    hovermode="x unified"
)

fig_pareto.show()

In [415]:
pareto_problematiques["problematique_affichage"] = (
    pareto_problematiques["problematique"]
    .str.slice(0, 25)
)

In [416]:
x=pareto_problematiques["problematique_affichage"]

In [417]:
# Export HTML
fig_pareto.write_html(
    FIGURES_DIR / "pareto_problematiques.html",
    include_plotlyjs=True,
    full_html=True
)

# Export PNG
try:
    fig_pareto.write_image(
        FIGURES_DIR / "pareto_problematiques.png",
        width=1800,
        height=1000,
        scale=2
    )
    print("Export PNG réussi.")
except Exception as e:
    print("Export PNG impossible :", e)

Export PNG réussi.


### Interprétation

Le diagramme de Pareto permet de visualiser la contribution cumulative des différentes
problématiques au volume total de lignes analytiques.

Les catégories sont classées par ordre décroissant de fréquence. La courbe cumulée
indique la proportion du volume total couverte par les catégories successives.

Le repère de 80 % constitue un seuil descriptif permettant d'identifier le nombre de
catégories représentant la majeure partie du volume observé.

Cette analyse peut aider à orienter les travaux ultérieurs de standardisation,
de documentation ou d'automatisation sur les catégories les plus fréquentes.

Elle ne permet toutefois pas, à elle seule, de déterminer les catégories les plus
complexes ou celles nécessitant le plus de temps de résolution.

In [424]:
# ============================================================
# Analyse des volumes et des temps de résolution par technicien
# ============================================================

stats_techniciens = (
    df_clean
    .groupby("technicien")
    .agg(
        nombre_lignes=("technicien", "size"),
        temps_moyen=("resolution_minutes", "mean"),
        temps_median=("resolution_minutes", "median"),
        temps_p95=("resolution_minutes", lambda x: x.quantile(0.95)),
        temps_min=("resolution_minutes", "min"),
        temps_max=("resolution_minutes", "max")
    )
    .reset_index()
    .sort_values("nombre_lignes", ascending=False)
)

# Pourcentage du volume total
stats_techniciens["pourcentage_volume"] = (
    stats_techniciens["nombre_lignes"]
    / stats_techniciens["nombre_lignes"].sum()
    * 100
)

stats_techniciens

,technicien,nombre_lignes,temps_moyen,temps_median,temps_p95,temps_min,temps_max,pourcentage_volume
1,Sofiane Aoues,601,3.893511,3.0,8.0,1.0,19.000000,88.382353
0,Loubna,79,12.326371,10.0,28.2,4.0,41.233333,11.617647


In [425]:
stats_techniciens.style.format({
    "temps_moyen": "{:.2f}",
    "temps_median": "{:.2f}",
    "temps_p95": "{:.2f}",
    "temps_min": "{:.2f}",
    "temps_max": "{:.2f}",
    "pourcentage_volume": "{:.2f}%"
})

,technicien,nombre_lignes,temps_moyen,temps_median,temps_p95,temps_min,temps_max,pourcentage_volume
1,Sofiane Aoues,601,3.89,3.00,8.00,1.00,19.00,88.38%
0,Loubna,79,12.33,10.00,28.20,4.00,41.23,11.62%


In [426]:
# Préparation des données
volume_techniciens = (
    stats_techniciens
    .sort_values("nombre_lignes", ascending=True)
)

# Graphique
fig_volume_techniciens = px.bar(
    volume_techniciens,
    x="nombre_lignes",
    y="technicien",
    orientation="h",
    text="nombre_lignes",
    title=(
        "Volume d’activité par technicien"
        "<br><sup>Nombre de lignes analytiques enregistrées</sup>"
    ),
    labels={
        "nombre_lignes": "Nombre de lignes analytiques",
        "technicien": "Technicien"
    },
    hover_data={
        "pourcentage_volume": ":.2f"
    }
)

fig_volume_techniciens.update_traces(
    textposition="outside"
)

fig_volume_techniciens.update_layout(
    template="plotly_white",
    height=450,
    width=1000,
    margin=dict(l=150, r=80, t=100, b=80),
    xaxis=dict(
        tickmode="linear",
        dtick=50
    )
)

fig_volume_techniciens.show()

In [427]:
fig_volume_techniciens.write_html(
    FIGURES_DIR / "volume_par_technicien.html"
)

fig_volume_techniciens.write_image(
    FIGURES_DIR / "volume_par_technicien.png",
    scale=2
)

In [428]:
import plotly.express as px
import pandas as pd

# 1. Transformation en format long pour garder Moyenne, Médiane et P95 par technicien
# (On s'assure d'utiliser le bon DataFrame 'stats_techniciens' avec les colonnes : technicien, temps_moyen, temps_median, temps_p95)
stats_techs_long = stats_techniciens.melt(
    id_vars=["technicien", "nombre_lignes"],
    value_vars=[
        "temps_moyen",
        "temps_median",
        "temps_p95"
    ],
    var_name="indicateur",
    value_name="duree_minutes"
)

noms_indicateurs = {
    "temps_moyen": "Moyenne",
    "temps_median": "Médiane",
    "temps_p95": "P95"
}

stats_techs_long["indicateur"] = (
    stats_techs_long["indicateur"]
    .map(noms_indicateurs)
)

# Optionnel : trier les techniciens pour un affichage propre
stats_techs_sorted = stats_techniciens.sort_values("temps_moyen", ascending=True)

# 2. Création du graphique groupé horizontal
fig_techs_temps = px.bar(
    stats_techs_long,
    x="duree_minutes",
    y="technicien",
    color="indicateur",
    barmode="group",
    orientation="h",
    text="duree_minutes",  # Affiche la valeur numérique exacte
    title=(
        "<b>Temps de résolution par Technicien (Moyenne vs Médiane vs P95)</b>"
        "<br><span style='font-size:12px; color:#64748b;'>Comparaison des indicateurs de temps de traitement par opérateur</span>"
    ),
    labels={
        "technicien": "Technicien",
        "duree_minutes": "Durée de résolution (minutes)",
        "indicateur": "Indicateur"
    },
    color_discrete_map={
        "Moyenne": "#3b82f6",  # Bleu pro
        "Médiane": "#10b981",  # Vert émeraude
        "P95": "#f87171"       # Rouge corail (seuil critique)
    },
    hover_data={
        "nombre_lignes": True,
        "duree_minutes": ":.2f"
    }
)

# 3. Personnalisation des textes et des barres
fig_techs_temps.update_traces(
    texttemplate='%{text:.1f} min',  # Formate le texte (1 chiffre après la virgule + "min")
    textposition="outside",          # Place le texte juste à côté de la barre
    cliponaxis=False,                # Empêche le texte d'être coupé s'il dépasse
    hovertemplate=(
        "<b>Technicien :</b> %{y}<br>"
        "<b>%{fullData.name} :</b> %{x:.2f} min<br>"
        "<b>Volume traité :</b> %{customdata[0]} lignes"
        "<extra></extra>"
    )
)

# 4. Mise en page aux standards professionnels
fig_techs_temps.update_layout(
    template="plotly_white",
    height=max(550, len(stats_techniciens) * 50),  # Ajuste dynamiquement la hauteur
    width=1250,
    title={
        "x": 0.0,
        "xanchor": "left"
    },
    xaxis=dict(
        title="Durée de résolution (minutes)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=stats_techs_sorted["technicien"].tolist(),
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    margin=dict(t=90, b=70, l=160, r=90),  # Marge droite élargie pour les textes "outside"
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.04,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    ),
    legend_title_text=""
)

fig_techs_temps.show()

# ============================================================
# 5. Exports autonomes (HTML et PNG)
# ============================================================

fig_techs_temps.write_html(
    FIGURES_DIR / "temps_resolution_par_technicien.html"
)

fig_techs_temps.write_image(
    FIGURES_DIR / "temps_resolution_par_technicien.png",
    scale=2
)

print(" Graphique des temps par technicien généré et exporté avec succès !")

 Graphique des temps par technicien généré et exporté avec succès !


### Interprétation
Ce graphique permet de comparer les tendances centrales (moyenne et médiane) ainsi que le seuil de haute dispersion (P95) des temps de résolution pour chaque technicien.

L'analyse conjointe de ces indicateurs permet de distinguer la rapidité de traitement habituelle (reflétée par la médiane, moins sensible aux valeurs extrêmes) de la lourdeur des cas exceptionnels (mis en évidence par le P95).

L'écart entre la moyenne et la médiane offre un premier aperçu de l'asymétrie de la charge de travail, signalant la présence potentielle de tickets particulièrement chronophages pour certains opérateurs.

Cette lecture croisée aide à évaluer l'homogénéité des pratiques au sein de l'équipe et à identifier les profils pouvant nécessiter un appui technique ou une répartition de charge ciblée.

Elle ne saurait suffire, à elle seule, à juger de la complexité intrinsèque des dossiers traités, celle-ci variant selon la nature des problématiques et le périmètre des comptes pris en charge.

In [429]:
# ============================================================
# Analyse des volumes et des temps de résolution par priorité
# ============================================================

stats_priorites = (
    df_clean
    .groupby("priorite")
    .agg(
        nombre_lignes=("priorite", "size"),
        temps_moyen=("resolution_minutes", "mean"),
        temps_median=("resolution_minutes", "median"),
        temps_p95=("resolution_minutes", lambda x: x.quantile(0.95)),
        temps_min=("resolution_minutes", "min"),
        temps_max=("resolution_minutes", "max")
    )
    .reset_index()
    .sort_values("nombre_lignes", ascending=False)
)

stats_priorites["pourcentage_volume"] = (
    stats_priorites["nombre_lignes"]
    / stats_priorites["nombre_lignes"].sum()
    * 100
)

stats_priorites

,priorite,nombre_lignes,temps_moyen,temps_median,temps_p95,temps_min,temps_max,pourcentage_volume
0,Basse,675,4.797852,4.000000,11.000000,1.000000,40.000000,99.264706
2,Moyenne,2,7.000000,7.000000,7.000000,7.000000,7.000000,0.294118
3,Urgente,2,10.000000,10.000000,15.400000,4.000000,16.000000,0.294118
1,Haute,1,41.233333,41.233333,41.233333,41.233333,41.233333,0.147059


In [430]:
stats_priorites.style.format({
    "temps_moyen": "{:.2f}",
    "temps_median": "{:.2f}",
    "temps_p95": "{:.2f}",
    "temps_min": "{:.2f}",
    "temps_max": "{:.2f}",
    "pourcentage_volume": "{:.2f}%"
})

,priorite,nombre_lignes,temps_moyen,temps_median,temps_p95,temps_min,temps_max,pourcentage_volume
0,Basse,675,4.80,4.00,11.00,1.00,40.00,99.26%
2,Moyenne,2,7.00,7.00,7.00,7.00,7.00,0.29%
3,Urgente,2,10.00,10.00,15.40,4.00,16.00,0.29%
1,Haute,1,41.23,41.23,41.23,41.23,41.23,0.15%


In [467]:
import plotly.express as px

# 1. Transformation en format long
stats_priorites_long = stats_priorites.melt(
    id_vars=["priorite", "nombre_lignes"],
    value_vars=[
        "temps_moyen",
        "temps_median",
        "temps_p95"
    ],
    var_name="indicateur",
    value_name="duree_minutes"
)

noms_indicateurs = {
    "temps_moyen": "Moyenne",
    "temps_median": "Médiane",
    "temps_p95": "P95"
}

stats_priorites_long["indicateur"] = (
    stats_priorites_long["indicateur"].map(noms_indicateurs)
)

# Ordre d'affichage selon le volume
ordre_priorites = stats_priorites["priorite"].tolist()

# 2. Création du graphique optimisé
fig_priorites_temps = px.bar(
    stats_priorites_long,
    x="duree_minutes",
    y="priorite",
    color="indicateur",
    barmode="group",
    orientation="h",
    text="duree_minutes",
    category_orders={"priorite": ordre_priorites},
    title=(
        "<b>Temps de résolution par niveau de priorité</b>"
        "<br><span style='font-size:12px; color:#64748b;'>Comparaison des tendances centrales et du P95 par criticité</span>"
    ),
    labels={
        "priorite": "Priorité",
        "duree_minutes": "Durée de résolution (minutes)",
        "indicateur": "Indicateur"
    },
    color_discrete_map={
        "Moyenne": "#3b82f6",  # Bleu pro
        "Médiane": "#10b981",  # Vert émeraude
        "P95": "#f87171"       # Rouge corail
    },
    hover_data={
        "nombre_lignes": True,
        "duree_minutes": ":.2f"
    }
)

# 3. Personnalisation avancée des barres et des infobulles
fig_priorites_temps.update_traces(
    texttemplate="%{text:.1f} min",
    textposition="outside",
    cliponaxis=False,
    marker=dict(
        line=dict(color="#1e293b", width=0.5)  # Contour fin pour le relief
    ),
    hovertemplate=(
        "<b>Priorité :</b> %{y}<br>"
        "<b>%{fullData.name} :</b> %{x:.2f} min<br>"
        "<b>Volume lignes :</b> %{customdata[0]}<br>"
        "<extra></extra>"
    )
)

# 4. Mise en page aux standards professionnels
fig_priorites_temps.update_layout(
    template="plotly_white",
    height=500,
    width=1250,
    title={
        "x": 0.0,
        "xanchor": "left"
    },
    margin=dict(l=130, r=100, t=90, b=60),
    xaxis=dict(
        title="Durée de résolution (minutes)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=ordre_priorites,
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.04,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    )
)

fig_priorites_temps.show()

Dans le rapport, il faudra préciser que :

la priorité Basse domine largement l’échantillon ;
les catégories Haute, Urgente et Moyenne ont des effectifs très faibles ;
les statistiques de ces catégories doivent être considérées avec prudence ;
aucune conclusion générale ne doit être tirée sur l’effet de la priorité sur le temps de résolution.

In [432]:
# ============================================================
# Évolution quotidienne des temps de résolution
# ============================================================

stats_quotidiennes = (
    df_clean
    .groupby("date_ticket")
    .agg(
        nombre_lignes=("resolution_minutes", "size"),
        temps_moyen=("resolution_minutes", "mean"),
        temps_median=("resolution_minutes", "median"),
        temps_p95=("resolution_minutes", lambda x: x.quantile(0.95))
    )
    .reset_index()
    .sort_values("date_ticket")
)

stats_quotidiennes.head()

,date_ticket,nombre_lignes,temps_moyen,temps_median,temps_p95
0,2026-06-23,5,17.646667,13.0,36.786667
1,2026-06-24,14,14.714286,9.0,36.750000
2,2026-06-25,13,10.792308,8.0,23.600000
3,2026-06-26,13,8.330769,8.0,15.800000
4,2026-06-29,13,7.611538,5.0,18.526667


In [433]:
stats_quotidiennes["moyenne_mobile_7j"] = (
    stats_quotidiennes["temps_moyen"]
    .rolling(window=7, min_periods=3)
    .mean()
)

stats_quotidiennes

,date_ticket,nombre_lignes,temps_moyen,temps_median,temps_p95,moyenne_mobile_7j
0,2026-06-23,5,17.646667,13.0,36.786667,NaN
1,2026-06-24,14,14.714286,9.0,36.750000,NaN
2,2026-06-25,13,10.792308,8.0,23.600000,14.384420
3,2026-06-26,13,8.330769,8.0,15.800000,12.871007
4,2026-06-29,13,7.611538,5.0,18.526667,11.819114
5,2026-06-30,7,13.428571,12.0,25.300000,12.087357
6,2026-07-01,4,8.250000,8.5,10.000000,11.539163
7,2026-07-02,8,7.375000,7.0,10.000000,10.071782
8,2026-07-03,13,6.384615,7.0,10.800000,8.881829
9,2026-07-06,10,7.600000,6.5,12.550000,8.425785


In [470]:
import plotly.graph_objects as go

# ============================================================
# Graphique de l'évolution quotidienne des temps de résolution
# ============================================================

fig_evolution_quotidienne = go.Figure()

# 1. Trace : Temps Moyen journalier
fig_evolution_quotidienne.add_trace(
    go.Scatter(
        x=stats_quotidiennes["date_ticket"],
        y=stats_quotidiennes["temps_moyen"],
        mode="lines+markers",
        name="Moyenne",
        line=dict(color="#3b82f6", width=1.5),
        marker=dict(size=5, color="#3b82f6"),
        hovertemplate=(
            "<b>Moyenne :</b> %{y:.2f} min<br>"
            "<extra></extra>"
        )
    )
)

# 2. Trace : Temps Médian journalier
fig_evolution_quotidienne.add_trace(
    go.Scatter(
        x=stats_quotidiennes["date_ticket"],
        y=stats_quotidiennes["temps_median"],
        mode="lines+markers",
        name="Médiane",
        line=dict(color="#10b981", width=1.5),
        marker=dict(size=5, color="#10b981"),
        hovertemplate=(
            "<b>Médiane :</b> %{y:.2f} min<br>"
            "<extra></extra>"
        )
    )
)

# 3. Mise en page aux standards professionnels
fig_evolution_quotidienne.update_layout(
    template="plotly_white",
    title=dict(
        text=(
            "<b>Évolution quotidienne des temps de résolution</b>"
            "<br><span style='font-size:12px; color:#64748b;'>Suivi journalier comparé de la moyenne et de la médiane</span>"
        ),
        font=dict(size=16, color="#1e293b", family="Arial"),
        x=0.0,
        y=0.92
    ),
    xaxis=dict(
        title="Date",
        tickformat="%d/%m/%Y",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="Durée de résolution (minutes)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    width=1300,
    height=550,
    hovermode="x unified",  # Affiche toutes les valeurs d'une même date dans une seule infobulle
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.06,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    ),
    margin=dict(t=100, b=60, l=70, r=40)
)

fig_evolution_quotidienne.show()

In [471]:
fig_evolution_quotidienne.write_html(
    FIGURES_DIR / "evolution_quotidienne_temps_resolution.html"
)

fig_evolution_quotidienne.write_image(
    FIGURES_DIR / "evolution_quotidienne_temps_resolution.png",
    scale=2
)

print("Graphique quotidien généré et exporté avec succès !")

Graphique quotidien généré et exporté avec succès !


In [473]:
import plotly.express as px

# ============================================================
# Volume quotidien associé (Optimisé)
# ============================================================

fig_volume_quotidien_associe = px.bar(
    stats_quotidiennes,
    x="date_ticket",
    y="nombre_lignes",
    title=(
        "<b>Volume quotidien des tickets</b>"
        "<br><span style='font-size:12px; color:#64748b;'>Indicateur de contexte pour interpréter les variations de durée</span>"
    ),
    labels={
        "date_ticket": "Date",
        "nombre_lignes": "Nombre de tickets"
    },
    text="nombre_lignes"
)

# Personnalisation des barres et des textes
fig_volume_quotidien_associe.update_traces(
    textposition="outside",
    cliponaxis=False,  # Évite que les textes du haut soient coupés
    marker=dict(
        color="#3b82f6",  # Bleu pro
        line=dict(color="#1d4ed8", width=0.8)  # Contour fin pour le relief
    ),
    hovertemplate=(
        "<b>Date :</b> %{x|%d/%m/%Y}<br>"
        "<b>Volume :</b> %{y} lignes<br>"
        "<extra></extra>"
    )
)

# Mise en page aux standards professionnels
fig_volume_quotidien_associe.update_layout(
    template="plotly_white",
    title={
        "x": 0.0,
        "xanchor": "left"
    },
    xaxis=dict(
        title="Date",
        tickformat="%d/%m/%Y",
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="Nombre de tcikets",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    width=1300,
    height=500,
    margin=dict(t=90, b=60, l=70, r=40)
)

fig_volume_quotidien_associe.show()

In [474]:
fig_volume_quotidien_associe.write_html(
    FIGURES_DIR / "volume_quotidien_contexte.html"
)

fig_volume_quotidien_associe.write_image(
    FIGURES_DIR / "volume_quotidien_contexte.png",
    scale=2
)

### Interprétation
L’évolution quotidienne permet d’observer les fluctuations des temps de résolution au cours de la période étudiée. La comparaison entre la moyenne, la médiane et le 95ᵉ percentile permet de distinguer les variations du traitement habituel des éventuels cas atypiques.

La moyenne mobile sur sept jours facilite l’identification d’une tendance générale en limitant l’influence des fluctuations journalières.

Le volume quotidien doit être examiné conjointement aux indicateurs de durée : une journée faiblement représentée peut produire des statistiques instables, tandis qu’une journée à fort volume peut davantage refléter les conditions habituelles de traitement.

Ces observations décrivent les variations temporelles des lignes analytiques enregistrées, mais ne permettent pas, à elles seules, d’attribuer les fluctuations à une évolution de la performance, de la complexité des demandes ou de la charge réelle de travail.


In [450]:
# ============================================================
# Évolution hebdomadaire des temps de résolution
# ============================================================

# Copie de travail
df_hebdo = df_clean.copy()

# Semaine de référence : lundi
df_hebdo["semaine"] = (
    df_hebdo["date_ticket"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

# Agrégation hebdomadaire
stats_hebdomadaires = (
    df_hebdo
    .groupby("semaine")
    .agg(
        nombre_lignes=("resolution_minutes", "size"),
        temps_moyen=("resolution_minutes", "mean"),
        temps_median=("resolution_minutes", "median"),
        temps_p95=("resolution_minutes", lambda x: x.quantile(0.95)),
        temps_min=("resolution_minutes", "min"),
        temps_max=("resolution_minutes", "max")
    )
    .reset_index()
    .sort_values("semaine")
)

# Moyenne mobile sur trois semaines
stats_hebdomadaires["moyenne_mobile_3_semaines"] = (
    stats_hebdomadaires["temps_moyen"]
    .rolling(window=3, min_periods=2)
    .mean()
)

stats_hebdomadaires

,semaine,nombre_lignes,temps_moyen,temps_median,temps_p95,temps_min,temps_max,moyenne_mobile_3_semaines
0,2026-06-22,45,12.062963,8.0,34.00,3.0,41.233333,NaN
1,2026-06-29,45,8.176667,7.0,19.00,2.0,28.000000,10.119815
2,2026-07-06,69,6.884058,6.0,12.00,3.0,13.000000,9.041229
3,2026-07-13,55,5.400000,4.0,13.30,1.0,16.000000,6.820242
4,2026-07-20,48,3.958333,4.0,7.00,2.0,10.000000,5.414130
5,2026-07-27,99,3.585859,3.0,6.10,2.0,9.000000,4.314731
6,2026-08-03,92,3.619565,3.0,7.00,2.0,10.000000,3.721252
7,2026-08-10,58,3.637931,3.0,6.00,2.0,8.000000,3.614452
8,2026-08-17,48,3.937500,4.0,6.00,2.0,10.000000,3.731665
9,2026-08-24,88,2.840909,3.0,4.00,2.0,6.000000,3.472113


In [451]:
print(
    "Période couverte :",
    stats_hebdomadaires["semaine"].min().strftime("%d/%m/%Y"),
    "→",
    stats_hebdomadaires["semaine"].max().strftime("%d/%m/%Y")
)

print(
    "Nombre de semaines observées :",
    len(stats_hebdomadaires)
)

Période couverte : 22/06/2026 → 14/09/2026
Nombre de semaines observées : 12


In [476]:
import plotly.graph_objects as go

# ============================================================
# Graphique de l'évolution hebdomadaire des temps de résolution
# ============================================================

fig_evolution_hebdomadaire = go.Figure()

# 1. Trace : Temps Moyen hebdomadaire
fig_evolution_hebdomadaire.add_trace(
    go.Scatter(
        x=stats_hebdomadaires["semaine"],
        y=stats_hebdomadaires["temps_moyen"],
        mode="lines+markers",
        name="Moyenne",
        line=dict(color="#3b82f6", width=2),
        marker=dict(size=6, color="#3b82f6"),
        hovertemplate=(
            "<b>Moyenne :</b> %{y:.2f} min<br>"
            "<extra></extra>"
        )
    )
)

# 2. Trace : Temps Médian hebdomadaire
fig_evolution_hebdomadaire.add_trace(
    go.Scatter(
        x=stats_hebdomadaires["semaine"],
        y=stats_hebdomadaires["temps_median"],
        mode="lines+markers",
        name="Médiane",
        line=dict(color="#10b981", width=2),
        marker=dict(size=6, color="#10b981"),
        hovertemplate=(
            "<b>Médiane :</b> %{y:.2f} min<br>"
            "<extra></extra>"
        )
    )
)

# 3. Mise en page aux standards professionnels
fig_evolution_hebdomadaire.update_layout(
    template="plotly_white",
    title=dict(
        text=(
            "<b>Évolution hebdomadaire des temps de résolution</b>"
            "<br><span style='font-size:12px; color:#64748b;'>Suivi hebdomadaire comparé de la moyenne et de la médiane</span>"
        ),
        font=dict(size=16, color="#1e293b", family="Arial"),
        x=0.0,
        y=0.92
    ),
    xaxis=dict(
        title="Semaine de référence",
        tickformat="%d/%m/%Y",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="Durée de résolution (minutes)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    width=1300,
    height=550,
    hovermode="x unified",  # Regroupe les valeurs de la semaine dans une seule infobulle
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.06,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    ),
    margin=dict(t=100, b=60, l=70, r=40)
)

fig_evolution_hebdomadaire.show()

In [478]:
import plotly.express as px

# ============================================================
# Volume hebdomadaire (Optimisé)
# ============================================================

fig_volume_hebdomadaire = px.bar(
    stats_hebdomadaires,
    x="semaine",
    y="nombre_lignes",
    text="nombre_lignes",
    title=(
        "<b>Volume hebdomadaire des tickets</b>"
        "<br><span style='font-size:12px; color:#64748b;'>Nombre total de lignes enregistrées par semaine de référence</span>"
    ),
    labels={
        "semaine": "Semaine de référence",
        "nombre_lignes": "Nombre de tickets"
    }
)

# Personnalisation des barres et des textes
fig_volume_hebdomadaire.update_traces(
    textposition="outside",
    cliponaxis=False,  # Empêche la troncature des textes du haut
    marker=dict(
        color="#3b82f6",  # Bleu pro
        line=dict(color="#1d4ed8", width=0.8)  # Contour fin pour le relief
    ),
    hovertemplate=(
        "<b>Semaine du :</b> %{x|%d/%m/%Y}<br>"
        "<b>Volume :</b> %{y} lignes<br>"
        "<extra></extra>"
    )
)

# Mise en page aux standards professionnels
fig_volume_hebdomadaire.update_layout(
    template="plotly_white",
    title={
        "x": 0.0,
        "xanchor": "left"
    },
    xaxis=dict(
        title="Semaine de référence",
        tickformat="%d/%m/%Y",
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="Nombre de tickets",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    width=1300,
    height=500,
    margin=dict(t=90, b=60, l=70, r=40)
)

fig_volume_hebdomadaire.show()

In [454]:
# ============================================================
# Exports
# ============================================================

fig_evolution_hebdomadaire.write_html(
    FIGURES_DIR / "evolution_hebdomadaire_temps_resolution.html"
)

fig_evolution_hebdomadaire.write_image(
    FIGURES_DIR / "evolution_hebdomadaire_temps_resolution.png",
    scale=2
)

fig_volume_hebdomadaire.write_html(
    FIGURES_DIR / "volume_hebdomadaire.html"
)

fig_volume_hebdomadaire.write_image(
    FIGURES_DIR / "volume_hebdomadaire.png",
    scale=2
)

print("Graphiques hebdomadaires générés et exportés avec succès !")

Graphiques hebdomadaires générés et exportés avec succès !


In [455]:
stats_hebdomadaires["niveau_fiabilite"] = np.select(
    [
        stats_hebdomadaires["nombre_lignes"] < 10,
        stats_hebdomadaires["nombre_lignes"] < 30
    ],
    [
        "Faible volume",
        "Volume modéré"
    ],
    default="Volume élevé"
)

stats_hebdomadaires[
    [
        "semaine",
        "nombre_lignes",
        "temps_moyen",
        "temps_median",
        "temps_p95",
        "niveau_fiabilite"
    ]
]

,semaine,nombre_lignes,temps_moyen,temps_median,temps_p95,niveau_fiabilite
0,2026-06-22,45,12.062963,8.0,34.00,Volume élevé
1,2026-06-29,45,8.176667,7.0,19.00,Volume élevé
2,2026-07-06,69,6.884058,6.0,12.00,Volume élevé
3,2026-07-13,55,5.400000,4.0,13.30,Volume élevé
4,2026-07-20,48,3.958333,4.0,7.00,Volume élevé
5,2026-07-27,99,3.585859,3.0,6.10,Volume élevé
6,2026-08-03,92,3.619565,3.0,7.00,Volume élevé
7,2026-08-10,58,3.637931,3.0,6.00,Volume élevé
8,2026-08-17,48,3.937500,4.0,6.00,Volume élevé
9,2026-08-24,88,2.840909,3.0,4.00,Volume élevé


Le regroupement hebdomadaire réduit l’effet des fluctuations quotidiennes et permet de mieux observer les tendances générales des temps de résolution.

La comparaison de la moyenne, de la médiane et du 95ᵉ percentile permet de distinguer le comportement habituel des durées et l’influence des cas les plus longs.

Le volume hebdomadaire constitue un indicateur de contexte indispensable : les semaines faiblement représentées peuvent produire des estimations instables.

Cette analyse décrit l’évolution des lignes analytiques enregistrées au cours de la période étudiée. Elle ne permet pas, à elle seule, de conclure à une amélioration ou à une dégradation de la performance opérationnelle.

In [456]:
# ============================================================
# Temps de résolution par problématique
# ============================================================

stats_problematiques = (
    df_clean
    .groupby("problematique")
    .agg(
        nombre_lignes=("resolution_minutes", "size"),
        temps_moyen=("resolution_minutes", "mean"),
        temps_median=("resolution_minutes", "median"),
        temps_p95=("resolution_minutes", lambda x: x.quantile(0.95)),
        temps_min=("resolution_minutes", "min"),
        temps_max=("resolution_minutes", "max")
    )
    .reset_index()
)

# Pourcentage du volume
stats_problematiques["pourcentage_volume"] = (
    stats_problematiques["nombre_lignes"]
    / stats_problematiques["nombre_lignes"].sum()
    * 100
)

# Tri selon la médiane
stats_problematiques = stats_problematiques.sort_values(
    "temps_median",
    ascending=True
)

stats_problematiques

,problematique,nombre_lignes,temps_moyen,temps_median,temps_p95,temps_min,temps_max,pourcentage_volume
2,Envoi Matériel,73,5.064384,3.0,13.92,1.0,41.233333,10.735294
5,Floorwalker,22,4.318182,3.0,8.90,2.0,13.000000,3.235294
6,Handover,22,2.818182,3.0,3.95,2.0,5.000000,3.235294
10,Intervention J+1,57,4.438596,3.0,12.00,2.0,19.000000,8.382353
13,Opening Assistance,28,2.892857,3.0,4.65,2.0,5.000000,4.117647
16,Swap Transporteur,95,3.747368,3.0,8.00,2.0,10.000000,13.970588
7,Installation,89,5.426966,4.0,10.00,2.0,35.000000,13.088235
3,Fermeture Définitive,8,5.250000,4.0,9.65,2.0,10.000000,1.176471
12,Maintenance,197,5.421743,4.0,13.20,2.0,40.000000,28.970588
8,Installation New Store,13,5.230769,4.0,9.40,3.0,13.000000,1.911765


In [457]:
stats_problematiques.style.format({
    "temps_moyen": "{:.2f}",
    "temps_median": "{:.2f}",
    "temps_p95": "{:.2f}",
    "temps_min": "{:.2f}",
    "temps_max": "{:.2f}",
    "pourcentage_volume": "{:.2f}%"
})

,problematique,nombre_lignes,temps_moyen,temps_median,temps_p95,temps_min,temps_max,pourcentage_volume
2,Envoi Matériel,73,5.06,3.00,13.92,1.00,41.23,10.74%
5,Floorwalker,22,4.32,3.00,8.90,2.00,13.00,3.24%
6,Handover,22,2.82,3.00,3.95,2.00,5.00,3.24%
10,Intervention J+1,57,4.44,3.00,12.00,2.00,19.00,8.38%
13,Opening Assistance,28,2.89,3.00,4.65,2.00,5.00,4.12%
16,Swap Transporteur,95,3.75,3.00,8.00,2.00,10.00,13.97%
7,Installation,89,5.43,4.00,10.00,2.00,35.00,13.09%
3,Fermeture Définitive,8,5.25,4.00,9.65,2.00,10.00,1.18%
12,Maintenance,197,5.42,4.00,13.20,2.00,40.00,28.97%
8,Installation New Store,13,5.23,4.00,9.40,3.00,13.00,1.91%


In [458]:
stats_problematiques["niveau_representation"] = np.select(
    [
        stats_problematiques["nombre_lignes"] < 5,
        stats_problematiques["nombre_lignes"] < 15
    ],
    [
        "Faible volume",
        "Volume modéré"
    ],
    default="Volume élevé"
)

In [459]:
# ============================================================
# Top 15 des problématiques les plus fréquentes
# ============================================================

top_problematiques = (
    stats_problematiques
    .nlargest(15, "nombre_lignes")
    .sort_values("temps_median", ascending=True)
)

In [479]:
import plotly.graph_objects as go

# ============================================================
# Graphique des temps de résolution par problématique (Optimisé)
# ============================================================

fig_problematiques_temps = go.Figure()

# 1. Barres : Temps Médian par problématique
fig_problematiques_temps.add_trace(
    go.Bar(
        y=top_problematiques["problematique"],
        x=top_problematiques["temps_median"],
        orientation="h",
        name="Médiane",
        text=top_problematiques["temps_median"],
        texttemplate="%{text:.1f} min",
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color="#3b82f6",  # Bleu pro
            line=dict(color="#1d4ed8", width=0.8)  # Contour fin pour le relief
        ),
        hovertemplate=(
            "<b>Problématique :</b> %{y}<br>"
            "<b>Médiane :</b> %{x:.2f} min"
            "<extra></extra>"
        )
    )
)

# 2. Marqueurs : P95 (Seuil critique / valeurs extrêmes)
fig_problematiques_temps.add_trace(
    go.Scatter(
        y=top_problematiques["problematique"],
        x=top_problematiques["temps_p95"],
        mode="markers",
        name="P95 (Seuil critique)",
        marker=dict(
            size=11,
            color="#f87171",  # Rouge corail élégant
            symbol="diamond",  # Forme distincte pour le P95
            line=dict(color="#991b1b", width=1)
        ),
        customdata=top_problematiques[["nombre_lignes", "pourcentage_volume"]],
        hovertemplate=(
            "<b>Problématique :</b> %{y}<br>"
            "<b>P95 :</b> %{x:.2f} min<br>"
            "<b>Volume :</b> %{customdata[0]} lignes<br>"
            "<b>Part du volume :</b> %{customdata[1]:.2f}%"
            "<extra></extra>"
        )
    )
)

# 3. Mise en page aux standards professionnels
fig_problematiques_temps.update_layout(
    template="plotly_white",
    title=dict(
        text=(
            "<b>Temps de résolution par problématique</b>"
            "<br><span style='font-size:12px; color:#64748b;'>Top 15 des problématiques — Comparaison de la médiane et du seuil critique (P95)</span>"
        ),
        font=dict(size=16, color="#1e293b", family="Arial"),
        x=0.0,
        y=0.92
    ),
    xaxis=dict(
        title="Durée de résolution (minutes)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="",
        categoryorder="total ascending",  # Assure un tri propre du top 15
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    width=1300,
    height=750,
    margin=dict(l=250, r=80, t=100, b=60),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.04,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    )
)

fig_problematiques_temps.show()

In [461]:
fig_problematiques_temps.write_html(
    FIGURES_DIR / "temps_resolution_par_problematique.html"
)

fig_problematiques_temps.write_image(
    FIGURES_DIR / "temps_resolution_par_problematique.png",
    scale=2
)

print("Graphique par problématique généré et exporté avec succès !")

Graphique par problématique généré et exporté avec succès !


L’analyse par problématique permet de comparer les temps de résolution associés aux différentes catégories de demandes.

La médiane décrit le temps de traitement habituel au sein de chaque catégorie, tandis que le P95 met en évidence les cas les plus longs.

La prise en compte du volume est indispensable : une problématique peu représentée peut présenter une durée élevée sans que cette observation soit suffisamment robuste pour être généralisée.

Les résultats permettent d’identifier les catégories susceptibles de nécessiter une documentation plus précise, une standardisation des procédures ou une analyse complémentaire.

Cette comparaison reste descriptive et ne permet pas, à elle seule, d’attribuer les différences de durée à la seule nature des problématiques.

In [462]:
# ============================================================
# Analyse croisée : volume et temps de résolution
# par problématique
# ============================================================

analyse_volume_duree = stats_problematiques.copy()

# Classement selon le volume
analyse_volume_duree["rang_volume"] = (
    analyse_volume_duree["nombre_lignes"]
    .rank(method="min", ascending=False)
)

# Classement selon la médiane
analyse_volume_duree["rang_temps_median"] = (
    analyse_volume_duree["temps_median"]
    .rank(method="min", ascending=False)
)

analyse_volume_duree = analyse_volume_duree.sort_values(
    "nombre_lignes",
    ascending=False
)

analyse_volume_duree.head(10)

,problematique,nombre_lignes,temps_moyen,temps_median,temps_p95,temps_min,temps_max,pourcentage_volume,niveau_representation,rang_volume,rang_temps_median
12,Maintenance,197,5.421743,4.0,13.20,2.0,40.000000,28.970588,Volume élevé,1.0,6.0
16,Swap Transporteur,95,3.747368,3.0,8.00,2.0,10.000000,13.970588,Volume élevé,2.0,12.0
7,Installation,89,5.426966,4.0,10.00,2.0,35.000000,13.088235,Volume élevé,3.0,6.0
2,Envoi Matériel,73,5.064384,3.0,13.92,1.0,41.233333,10.735294,Volume élevé,4.0,12.0
10,Intervention J+1,57,4.438596,3.0,12.00,2.0,19.000000,8.382353,Volume élevé,5.0,12.0
11,Intervention sur Site,32,4.687500,4.0,8.90,2.0,10.000000,4.705882,Volume élevé,6.0,6.0
13,Opening Assistance,28,2.892857,3.0,4.65,2.0,5.000000,4.117647,Volume élevé,7.0,12.0
5,Floorwalker,22,4.318182,3.0,8.90,2.0,13.000000,3.235294,Volume élevé,8.0,12.0
6,Handover,22,2.818182,3.0,3.95,2.0,5.000000,3.235294,Volume élevé,8.0,12.0
15,Remplacement Matériel,17,7.529412,6.0,16.60,4.0,19.000000,2.500000,Volume élevé,10.0,2.0


In [481]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Optionnel : Tri par volume décroissant pour un affichage propre
df_viz = analyse_volume_duree.sort_values("nombre_lignes", ascending=True)

# 2. Création de subplots (1 ligne, 2 colonnes partageant l'axe Y des problématiques)
fig_volume_duree = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    subplot_titles=(
        "<b>Volume de tickets</b>", 
        "<b>Temps médian de résolution (min)</b>"
    ),
    horizontal_spacing=0.15
)

# --- Trace 1 : Volume (Barres bleues à gauche) ---
fig_volume_duree.add_trace(
    go.Bar(
        y=df_viz["problematique"],
        x=df_viz["nombre_lignes"],
        orientation="h",
        name="Volume",
        text=df_viz["nombre_lignes"],
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color="#3b82f6",  # Bleu pro
            line=dict(color="#1d4ed8", width=0.8)
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Volume : %{x} lignes<br>"
            "<extra></extra>"
        )
    ),
    row=1, col=1
)

# --- Trace 2 : Temps Médian (Barres vertes à droite) ---
fig_volume_duree.add_trace(
    go.Bar(
        y=df_viz["problematique"],
        x=df_viz["temps_median"],
        orientation="h",
        name="Temps Médian",
        text=df_viz["temps_median"],
        texttemplate="%{text:.1f} min",
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color="#10b981",  # Vert émeraude
            line=dict(color="#047857", width=0.8)
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Temps Médian : %{x:.2f} min<br>"
            "<extra></extra>"
        )
    ),
    row=1, col=2
)

# 3. Mise en page globale aux standards professionnels
fig_volume_duree.update_layout(
    template="plotly_white",
    title=dict(
        text=(
            "<b>Analyse croisée : Volume vs Temps médian par problématique</b>"
            "<br><span style='font-size:12px; color:#64748b;'>Évaluation conjointe de la charge de travail et de la durée de traitement par catégorie</span>"
        ),
        font=dict(size=16, color="#1e293b", family="Arial"),
        x=0.0,
        y=0.92
    ),
    xaxis=dict(
        title="Nombre de tickets",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    xaxis2=dict(
        title="Durée médiane (minutes)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#e2e8f0",
        linecolor="#cbd5e1"
    ),
    yaxis=dict(
        title="",
        showgrid=False,
        linecolor="#cbd5e1"
    ),
    width=1400,
    height=max(600, len(df_viz) * 35),
    margin=dict(l=240, r=80, t=110, b=60),
    showlegend=False  # Les titres de colonnes suffisent amplement
)

fig_volume_duree.show()

In [482]:
# Valeurs de référence
volume_median_categories = analyse_volume_duree["nombre_lignes"].median()
temps_median_global = df_clean["resolution_minutes"].median()

fig_volume_duree.add_vline(
    x=volume_median_categories,
    line_dash="dash",
    annotation_text="Volume médian des catégories",
    annotation_position="top"
)

fig_volume_duree.add_hline(
    y=temps_median_global,
    line_dash="dash",
    annotation_text="Médiane globale",
    annotation_position="right"
)

fig_volume_duree.show()

In [465]:
fig_volume_duree.write_html(
    FIGURES_DIR / "volume_temps_par_problematique.html"
)

fig_volume_duree.write_image(
    FIGURES_DIR / "volume_temps_par_problematique.png",
    scale=2
)

print("Graphique volume / durée généré et exporté avec succès !")

Graphique volume / durée généré et exporté avec succès !


Le graphique met en relation le volume d’activité et le temps médian de résolution pour chaque problématique.

Les catégories situées dans la zone de volume élevé et de temps médian élevé peuvent constituer des sujets prioritaires pour une analyse opérationnelle complémentaire.

Les catégories présentant un faible volume doivent être interprétées avec prudence, car leurs indicateurs peuvent être sensibles à quelques observations individuelles.

Les lignes de référence facilitent la comparaison descriptive entre les catégories, mais ne constituent pas des seuils de performance.

Cette représentation ne permet pas, à elle seule, d’établir une relation causale entre la nature d’une problématique et le temps de résolution.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# --- ÉTAPE 1 : Création et nettoyage du DataFrame ---
# (Remplacez cette partie par votre propre chargement de données, ex: pd.read_csv('votre_fichier.csv'))

# Simulation de données brutes
np.random.seed(42)
n_lignes = 500
jours_bruts = np.random.choice(['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche'], n_lignes)
heures_brutes = np.random.randint(8, 19, n_lignes) # Heures de bureau entre 8h et 18h

df = pd.DataFrame({
    'Jour ouverture': jours_bruts,
    'Heure ouverture analytique': heures_brutes
})

# Nettoyage 1 : Suppression des doublons ou valeurs manquantes si nécessaire
df = df.dropna(subset=['Jour ouverture', 'Heure ouverture analytique'])

# Nettoyage 2 : Définition d'une catégorie ordonnée pour les jours de la semaine 
# (Indispensable pour éviter un tri alphabétique chaotique sur l'axe Y)
ordre_jours = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
df['Jour ouverture'] = pd.Categorical(df['Jour ouverture'], categories=ordre_jours, ordered=True)


# --- ÉTAPE 2 : Génération de la Heatmap avec Plotly ---

fig = px.density_heatmap(
    df,
    x='Heure ouverture analytique',
    y='Jour ouverture',
    text_auto=True,  # Affiche directement le nombre de tickets dans les cases
    title="Volume de tickets par Jour et par Heure (Heatmap d'activité)",
    labels={
        'Heure ouverture analytique': "Heure de la journée",
        'Jour ouverture': "Jour de la semaine",
        'count': "Nombre de tickets"
    },
    color_continuous_scale='Blues'  # Palette de couleurs (ex: 'Viridis', 'Blues', 'Cividis')
)

# --- ÉTAPE 3 : Personnalisation de l'affichage ---
fig.update_layout(
    xaxis=dict(
        tickmode='linear',
        dtick=1  # Force l'affichage de chaque heure sur l'axe X
    ),
    xaxis_title="Heure d'ouverture",
    yaxis_title="Jour de la semaine",
    title_font_size=16
)

# Affichage du graphique dans notebook Jupyter
fig.show()

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

# --- ÉTAPE 1 : Création et nettoyage du DataFrame ---
# (Remplacez cette partie par le chargement de votre dataframe réel, ex: df = pd.read_csv('votre_fichier.csv'))

# Simulation de données pour l'exemple
np.random.seed(42)
n_lignes = 400
clients = np.random.choice(['Client A', 'Client B', 'Client C', 'Client D'], n_lignes)
problematiques = np.random.choice(['Panne Réseau', 'Erreur Logiciel', 'Mot de passe', 'Matériel cassé'], n_lignes)
 techniciens = np.random.choice(['Alice', 'Bob', 'Charlie'], n_lignes)

df = pd.DataFrame({
    'Client Normalisé': clients,
    'Problématique Normalisée': problematiques,
    'Technicien Normalisé': techniciens
})

# Nettoyage : Suppression des lignes comportant des valeurs manquantes sur ces colonnes
df = df.dropna(subset=['Client Normalisé', 'Problématique Normalisée', 'Technicien Normalisé'])

# Nettoyage des chaînes de caractères (suppression des espaces superflus potentiels)
for col in ['Client Normalisé', 'Problématique Normalisée', 'Technicien Normalisé']:
    df[col] = df[col].astype(str).str.strip()


# --- ÉTAPE 2 : Préparation des données pour le Sunburst ---
# Le Sunburst a besoin de compter les occurrences de chaque combinaison hiérarchique
df_grouped = df.groupby(['Client Normalisé', 'Problématique Normalisée', 'Technicien Normalisé']).size().reset_index(name='Nombre_Tickets')


# --- ÉTAPE 3 : Génération du Sunburst Interactif avec Plotly ---
fig = px.sunburst(
    df_grouped,
    path=['Client Normalisé', 'Problématique Normalisée', 'Technicien Normalisé'],
    values='Nombre_Tickets',
    title="<b>Vue hiérarchique des tickets : Client $\rightarrow$ Problématique $\rightarrow$ Technicien</b>",
    color='Nombre_Tickets',
    color_continuous_scale='Blues', # Palette de couleurs
)

# Personnalisation de l'affichage pour plus de lisibilité
fig.update_layout(
    title_font_size=16,
    margin = dict(t=50, l=0, r=0, b=0)
)

# Affichage du graphique dans le notebook Jupyter
fig.show()

IndentationError: unexpected indent (771936265.py, line 13)

In [ ]:
df_clean.head(
)()

NameError: name 'df_clean' is not defined